<div style="background-color: #F4F6F7; padding: 20px; border-radius: 8px; font-family: 'Arial', sans-serif; color: #2E4053;">
  <!-- Logo centrado -->
  <div style="text-align: center; margin-bottom: 15px;">
    <img src="https://drive.google.com/uc?export=view&id=1kjzXfjTiieYAd4azw5bh4UfEg91gUIdh" alt="Logo Institucional" width="250" />
  </div>

  <!-- Título principal -->
  <h1 style="font-size: 36px; font-weight: bold; text-align: right;">
    <strong>Estimación del gradiente geotérmico a partir de imágenes multiespectrales del LANDSAT 7 y base de datos tabular mediante un sistema con atención intramodal e intermodal residual, además de weighted sampler, selección de variables y clipping de atípicos</strong>
  </h1>

  <!-- Datos del estudiante -->
  <p style="font-size: 22px; text-align: center; margin: 5px 0;">
    <strong>Nombre:</strong> Carlos Enrique Montilla Morales &nbsp;|&nbsp;
    <strong>Institución:</strong> Universidad Tecnológica de Pereira  &nbsp;|&nbsp;
    <strong>Grupo de investigación en Automática </strong>
  </p>

  <hr style="border: 1px solid #ABB2B9; margin: 10px 0;" />

  <!-- Descripción breve -->
  <p style="font-size: 18px; text-align: center; font-style: italic; margin: 5px 0;">
    Este cuaderno es la primera prueba donde se entrena un modelo multimodal mediante el uso de imágenes satelitales multiespectrales del LANDSAT7 y una base de datos que contienen diversas variables geológicas y geofísicas para estimar el gradiente geotérmico en Colombia. Incluye: ingestión y preprocesado de mosaicos (manejo de NoData), creación de muestras/parches, estrategia de fine-tuning, evaluación con métricas de regresión R2, MAE y RMSE y exportación de predicciones y checkpoints para análisis geoespacial.

  </p>

  <hr style="border: 1px solid #ABB2B9; margin: 10px 0;" />
</div>


# **1. Carga de las librerías, imágenes y la base de datos**

In [ ]:
!pip install -q --upgrade pip
!pip install -q rasterio==1.4 pyproj rioxarray
!pip install -q torchvision tqdm scikit-image pandas
!pip install -q torch-optimizer timm optuna

import warnings
warnings.filterwarnings(
    "ignore",
    message="This overload of addcmul_ is deprecated"
)

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
!gdown 1OIJY8QtxRYr4HW504Awxama22OAuBJID # Base de datos cruda
#!gdown 1oZAsFy_yDXnJUbUQbuIMlIS-SFaEhd_X #Geotiffs completos
#!gdown 1CnCNvGQSoWNEIbNz-j5bIa_COHpK15-l #Geotiffs completos 256
#!gdown 1wfjvuzJcSUH60iDi4wlvAIvOWLcdenKK #Geotiffs completos 224 y 9 bandas
#!gdown 1r2c3X8AlXOReL611kt7FqdRUu18jLsKf #Geotiff completos 224 y 10 bandas
!gdown 1r5wKeI-GGlznDuOljIGXu08egrgfY8UN #Geotiff completos 224 y 10 bandas con knn

!unzip -q LANDSAT_GEOTIFF_10_bands_knn.zip -d LANDSAT_GEOTIFF

In [ ]:
import os, re, time, joblib, json, shutil, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

import rasterio
from scipy.spatial import cKDTree
from skimage.transform import resize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch_optimizer as optim
import timm
import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import NearestNeighbors

import geopandas as gpd

BASE_DIR = Path("/kaggle/working/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF")
CSV_BASE = Path("/kaggle/working/data_prep.csv")
TARGET_COL = "Apparent Geothermal Gradient (°C/Km)"
OUT_DIR = Path("/kaggle/working/run_convnext_wsmoter_attention")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OPTUNA_TMP_DIR = Path("/kaggle/temp/optuna_trials")
TOP_K_CKPT_DIR = OUT_DIR / "optuna_top5_checkpoints"
TOP_K = 5

OPTUNA_TMP_DIR.mkdir(parents=True, exist_ok=True)
TOP_K_CKPT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = (224, 224)
FILL_METHOD_IMAGE = 'knn'
K_NEIGHBORS = 8
IMAGE_NORM = 'zscore'
TAB_IMPUTE_METHOD = 'knn'

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
NUM_WORKERS = 2

EPOCHS_HEAD = 5
EPOCHS_BACKBONE = 5
EPOCHS_FULL = 6

LR_HEAD = 1e-4
LR_FINE = 1e-5
LR_FULL = 1e-6
WEIGHT_DECAY = 1e-3
MAX_GRAD_NORM = 1.0

SEED = 42

# WSMOTER tabular
USE_WSMOTER_TABULAR = True
WSMOTER_MULTIPLIER = 1.3      # 1.35 = crea +35% muestras sintéticas en train
WSMOTER_K_NEIGHBORS = 5
WSMOTER_WEIGHT_POWER = 2     # >1 aumenta la probabilidad de generar extremos
WSMOTER_ID_MODE = "anchor"     # La imagen se conserva usando el ID de la muestra ancla
LDS_SIGMA = 5
LDS_N_BINS = 15

# Optuna
USE_OPTUNA = True
OPTUNA_N_TRIALS = 16
OPTUNA_TIMEOUT = None          # Ejemplo: 7200 para limitar a 2 horas
OPTUNA_STUDY_NAME = "convnext9_multimodal_optuna"
OPTUNA_DIR = OUT_DIR / "optuna"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

# Aumentación espacial para imágenes
USE_IMAGE_AUGMENTATION = True
AUG_P_HFLIP = 0.50
AUG_P_VFLIP = 0.50
AUG_P_ROT90 = 0.50
AUG_P_NOISE = 0.35
AUG_NOISE_SIGMA_IMAGE = 0.025
AUG_NOISE_SIGMA_THERMAL = 0.020
AUG_CLIP_VALUE = 5.0


# 9 bandas: sin Band2.
# 10 bandas: con Band2.
BAND_ORDER_9 = [
    "Band61_TOA_BT",
    "Band1_SRF_REF",
    "Band3_SRF_REF",
    "Band4_SRF_REF",
    "Band5_SRF_REF",
    "Band7_SRF_REF",
    "NDVI_SRF",
    "Band3_div_Band1",
    "Band5_div_Band7"
]

BAND_ORDER_10 = [
    "Band61_TOA_BT",
    "Band1_SRF_REF",
    "Band2_SRF_REF",
    "Band3_SRF_REF",
    "Band4_SRF_REF",
    "Band5_SRF_REF",
    "Band7_SRF_REF",
    "NDVI_SRF",
    "Band3_div_Band1",
    "Band5_div_Band7"
]

def detect_band_config(base_dir):
    tif_files = sorted(Path(base_dir).glob("*.tif"))
    if not tif_files:
        raise FileNotFoundError(f"No se encontraron GeoTIFF en {base_dir}")

    with rasterio.open(tif_files[0]) as ds:
        n_bands = ds.count

    if n_bands == 9:
        band_order = BAND_ORDER_9
        thermal_idxs = [0, 6, 7, 8]      # Band61, NDVI, B3/B1, B5/B7
    elif n_bands == 10:
        band_order = BAND_ORDER_10
        thermal_idxs = [0, 7, 8, 9]      # Band61, NDVI, B3/B1, B5/B7
    else:
        raise ValueError(
            f"Se detectaron {n_bands} bandas. Este notebook espera 9 o 10 bandas."
        )

    return n_bands, band_order, thermal_idxs

N_IMAGE_CHANNELS, BAND_ORDER, THERMAL_DERIVED_IDXS = detect_band_config(BASE_DIR)
IMAGE_IDXS = list(range(N_IMAGE_CHANNELS))
N_THERMAL_DERIVED_CHANNELS = len(THERMAL_DERIVED_IDXS)

# Backbone visual
CONVNEXT_BACKBONE = "convnext_tiny"
PRETRAINED_CONVNEXT = True

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("Device:", DEVICE)
print("USE_WSMOTER_TABULAR =", USE_WSMOTER_TABULAR)
print("USE_IMAGE_AUGMENTATION =", USE_IMAGE_AUGMENTATION)
print("N_IMAGE_CHANNELS =", N_IMAGE_CHANNELS)
print("BAND_ORDER =", BAND_ORDER)
print("THERMAL_DERIVED_IDXS =", THERMAL_DERIVED_IDXS)
print("N_THERMAL_DERIVED_CHANNELS =", N_THERMAL_DERIVED_CHANNELS)

In [ ]:
path = BASE_DIR / "LANDSAT7_1.tif"   # cambia por un ID real

with rasterio.open(path) as ds:
    print("Número de bandas:", ds.count)
    print("Shape:", ds.height, ds.width)
    print("Descriptions:", ds.descriptions)
    print("Indexes:", ds.indexes)

    for i in ds.indexes:
        print(f"\n--- Banda {i} ---")
        print("Descripción:", ds.descriptions[i-1])
        print("Tags:", ds.tags(i))

In [ ]:
# Cell 2

def read_geotiff(path: Path):
    with rasterio.open(str(path)) as ds:
        arr = ds.read().astype(np.float32)
        nodata = ds.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan
        return arr, ds.meta

def split_optical_thermal(arr):
    """
    Conserva los nombres optical/thermal para modificar lo mínimo del notebook.

    optical  -> stack completo de N_IMAGE_CHANNELS para ConvNeXt-Tiny.
    thermal  -> rama auxiliar con Band61 + NDVI + Band3/Band1 + Band5/Band7.
    """
    if arr.shape[0] < N_IMAGE_CHANNELS:
        raise ValueError(
            f"El GeoTIFF tiene {arr.shape[0]} bandas, pero este modelo espera "
            f"{N_IMAGE_CHANNELS}. Revisa que estés usando los recortes correctos."
        )

    optical = arr[IMAGE_IDXS].astype(np.float32)
    thermal = arr[THERMAL_DERIVED_IDXS].astype(np.float32)

    return optical, thermal

def fill_missing_median(arr):
    out = arr.copy()
    for b in range(out.shape[0]):
        band = out[b]
        mask = np.isnan(band)
        if mask.any():
            med = np.nanmedian(band)
            band[mask] = med if not np.isnan(med) else 0.0
        out[b] = band
    return out

def fill_missing_knn_mean(arr, k=8):
    out = arr.copy()
    for b in range(out.shape[0]):
        band = out[b]
        nan = np.isnan(band)
        if not nan.any():
            continue

        ys, xs = np.where(~nan)

        if len(ys) == 0:
            band[nan] = 0.0
            out[b] = band
            continue

        tree = cKDTree(np.c_[ys, xs])
        ys_n, xs_n = np.where(nan)

        _, idx = tree.query(np.c_[ys_n, xs_n], k=min(k, len(ys)))

        if np.ndim(idx) == 1:
            band[ys_n, xs_n] = band[ys[idx], xs[idx]]
        else:
            band[ys_n, xs_n] = band[ys[idx], xs[idx]].mean(axis=1)

        out[b] = band

    return out

def resize_multiband(arr, target_size=TARGET_SIZE):
    bands, H, W = arr.shape
    Ht, Wt = target_size

    if (H, W) == (Ht, Wt):
        return arr.astype(np.float32)

    arr_t = np.transpose(arr, (1, 2, 0))

    resized = resize(
        arr_t,
        (Ht, Wt, bands),
        order=1,
        preserve_range=True,
        anti_aliasing=True
    )

    return np.transpose(resized, (2, 0, 1)).astype(np.float32)

def _update_running_channel_stats(arr, sums, sq_sums, counts):
    """
    Estadística por canal sin concatenar todos los píxeles en memoria.
    arr: [C, H, W]
    """
    for c in range(arr.shape[0]):
        vals = arr[c].reshape(-1)
        vals = vals[np.isfinite(vals)]

        if vals.size == 0:
            continue

        sums[c] += vals.sum(dtype=np.float64)
        sq_sums[c] += np.square(vals, dtype=np.float64).sum(dtype=np.float64)
        counts[c] += vals.size

    return sums, sq_sums, counts

def _finalize_running_stats(sums, sq_sums, counts):
    mean = sums / np.maximum(counts, 1)
    var = (sq_sums / np.maximum(counts, 1)) - mean**2
    var = np.maximum(var, 1e-12)
    std = np.sqrt(var) + 1e-8

    return mean.astype(np.float32), std.astype(np.float32)

def compute_image_stats_train(df_split):
    sums = np.zeros(N_IMAGE_CHANNELS, dtype=np.float64)
    sq_sums = np.zeros(N_IMAGE_CHANNELS, dtype=np.float64)
    counts = np.zeros(N_IMAGE_CHANNELS, dtype=np.float64)

    for _, row in tqdm(df_split.iterrows(), total=len(df_split), desc=f"Image stats {N_IMAGE_CHANNELS}ch"):
        arr, _ = read_geotiff(BASE_DIR / f"LANDSAT7_{int(row.ID)}.tif")
        image, _ = split_optical_thermal(arr)

        image = fill_missing_median(image)
        #image = fill_missing_knn_mean(image)
        image = resize_multiband(image, TARGET_SIZE)

        sums, sq_sums, counts = _update_running_channel_stats(
            image, sums, sq_sums, counts
        )

    return _finalize_running_stats(sums, sq_sums, counts)

def compute_thermal_stats_train(df_split):
    sums = np.zeros(N_THERMAL_DERIVED_CHANNELS, dtype=np.float64)
    sq_sums = np.zeros(N_THERMAL_DERIVED_CHANNELS, dtype=np.float64)
    counts = np.zeros(N_THERMAL_DERIVED_CHANNELS, dtype=np.float64)

    for _, row in tqdm(df_split.iterrows(), total=len(df_split), desc="Thermal/derived stats"):
        arr, _ = read_geotiff(BASE_DIR / f"LANDSAT7_{int(row.ID)}.tif")
        _, thermal = split_optical_thermal(arr)

        thermal = fill_missing_median(thermal)
        #thermal = fill_missing_knn_mean(thermal)
        thermal = resize_multiband(thermal, TARGET_SIZE)

        sums, sq_sums, counts = _update_running_channel_stats(
            thermal, sums, sq_sums, counts
        )

    return _finalize_running_stats(sums, sq_sums, counts)


In [ ]:
# Cell 3

df_base = pd.read_csv(CSV_BASE, delimiter=';')
df_base = df_base.drop(columns=['Unnamed: 23'])
# df_base = df_base[['Latitude', 'Longitude', 'Elevation (m)',
#        'Curie Depth (Km)','Moho Depth (m)',
#        'Strike-slip Fault', 'Reverse or Thrust Fault',
#        'Right-lateral Fault', 'Nearest Basement',
#        'Normal Fault', 'Active Fault','Magnetic Anomaly (nT)',
#        'Vertical Gravity Gradient (E)', 'Free Air Anomaly (mGal)',
#        'Bouguer Anomaly (mGal)', 'ID', 'Apparent Geothermal Gradient (°C/Km)']]
df = df_base.copy()
df['ID'] = df['ID'].astype(str).str.extract(r'(\d+)').astype(int)
df['tif_name'] = df['ID'].apply(lambda x: f"LANDSAT7_{x}.tif")
df = df[df['tif_name'].apply(lambda x: (BASE_DIR/x).exists())].reset_index(drop=True)

FINAL_CSV = OUT_DIR / "final_dataset.csv"
df.to_csv(FINAL_CSV, index=False)
print("CSV final:", FINAL_CSV)

In [ ]:
# Cargar datos
df_final = pd.read_csv(FINAL_CSV)

extreme_gradient_threshold = df_final['Apparent Geothermal Gradient (°C/Km)'].quantile(0.99)
print(f"Extreme gradient threshold (97.5th percentile): {extreme_gradient_threshold}")

lowest_gradient_threshold = df_final['Apparent Geothermal Gradient (°C/Km)'].quantile(0.01)
print(f"Lowest gradient threshold (0.25st percentile): {lowest_gradient_threshold}")

df_final = df_final[df_final['Apparent Geothermal Gradient (°C/Km)'] <= extreme_gradient_threshold]
df_final = df_final[df_final['Apparent Geothermal Gradient (°C/Km)'] >= lowest_gradient_threshold]

df_final.info()

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter1d


def bins_freedman_diaconis(x, n):
    x = np.asarray(x, dtype=float)

    q1 = np.percentile(x, 25)
    q3 = np.percentile(x, 75)
    iqr = q3 - q1

    h = 2 * iqr / (n ** (1 / 3))

    if (not np.isfinite(h)) or h <= 0:
        return 10

    k = np.ceil((x.max() - x.min()) / h)

    if (not np.isfinite(k)) or k < 2:
        return 2

    return int(k)


def add_gradient_weights_lds(
    df,
    target_col,
    weight_col="Gradient Weight",
    sigma=15,
    n_bins=LDS_N_BINS
):
    """
    Calcula pesos LDS solo para el dataframe recibido.

    Para evitar fuga de información, esta función debe aplicarse al split
    de entrenamiento, no a df_final completo.
    """
    df = df.copy()

    y = df[target_col].to_numpy(dtype=float)

    #n_bins = bins_freedman_diaconis(y, len(y))
    hist, bin_edges = np.histogram(y, bins=n_bins)

    smoothed_hist = gaussian_filter1d(hist.astype(float), sigma=sigma)
    smoothed_hist = np.maximum(smoothed_hist, 1e-8)

    bin_ids = np.digitize(y, bin_edges[:-1]) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    density = smoothed_hist[bin_ids]

    weights = 1.0 / density
    weights = weights / weights.mean()

    df[weight_col] = weights.astype(np.float32)

    print(f"[LDS] Pesos calculados solo sobre train | sigma={sigma} | bins={n_bins}")
    print(f"[LDS] Max weight: {weights.max():.6f}")
    print(f"[LDS] Min weight: {weights.min():.6f}")
    print(f"[LDS] Mean weight: {weights.mean():.6f}")
    print(f"[LDS] Median weight: {np.median(weights):.6f}")

    return df


In [ ]:
# Dataset tabular base.
# Gradient Weight NO se calcula aquí para evitar usar validación/test en la distribución LDS.
# Latitude y Longitude se usarán como entrada del modelo.
# Latitude_raw y Longitude_raw se conservan sin normalizar para mapas.
GEO_LAT_COL = "Latitude_raw"
GEO_LON_COL = "Longitude_raw"

df_final = df_final[['Latitude', 'Longitude', 'Elevation (m)',
       'Curie Depth (Km)','Moho Depth (m)',
       'Strike-slip Fault', 'Reverse or Thrust Fault',
       'Right-lateral Fault', 'Nearest Basement',
       'Normal Fault', 'Active Fault','Magnetic Anomaly (nT)',
       'Vertical Gravity Gradient (E)', 'Free Air Anomaly (mGal)',
       'Bouguer Anomaly (mGal)', 'ID',
       'Apparent Geothermal Gradient (°C/Km)']].copy()

for c in df_final.columns:
    if c not in ['ID']:
        df_final[c] = pd.to_numeric(df_final[c], errors='coerce')

# Coordenadas reales para cartografía.
# Estas columnas NO entran al modelo ni al escalador.
df_final[GEO_LAT_COL] = df_final["Latitude"].copy()
df_final[GEO_LON_COL] = df_final["Longitude"].copy()

# IMPORTANTE:
# No se eliminan Latitude ni Longitude de tabular_cols.
# Por tanto, sí entran al modelo y se normalizan con StandardScaler.
drop_cols = ['ID', TARGET_COL, 'tif_name', 'Gradient Weight', GEO_LAT_COL, GEO_LON_COL]
drop_cols = [c for c in drop_cols if c in df_final.columns]

# Columnas que no se escalan.
# Latitude y Longitude NO están aquí, porque sí se quieren normalizadas para el modelo.
NO_SCALE_COLS = ['Volcanic domain', 'Volcanic weight']
NO_SCALE_COLS = [c for c in NO_SCALE_COLS if c in df_final.columns]

tabular_cols = [
    c for c in df_final.columns
    if c not in drop_cols
    and pd.api.types.is_numeric_dtype(df_final[c])
    and not df_final[c].isna().all()
]

scale_cols = [c for c in tabular_cols if c not in NO_SCALE_COLS]

print("Columnas tabulares:", tabular_cols)
print("Columnas escaladas:", scale_cols)
print("Columnas NO escaladas:", NO_SCALE_COLS)
print("Coordenadas para mapas:", GEO_LON_COL, GEO_LAT_COL)

df_final = df_final.dropna(
    subset=tabular_cols + [TARGET_COL, 'ID', GEO_LAT_COL, GEO_LON_COL]
).reset_index(drop=True)

print("Registros tras eliminar NaNs:", len(df_final))

# SPLIT train / val / test
# IMPORTANTE:
# El oversampling se aplica solo al train, nunca a validación ni test.
train_df, temp_df = train_test_split(
    df_final, test_size=0.20, random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42
)

train_df = train_df.reset_index(drop=True).copy()
val_df   = val_df.reset_index(drop=True).copy()
test_df  = test_df.reset_index(drop=True).copy()

print(f"Split original -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Copias crudas de los splits.
# Optuna las usa para reconstruir train/validation en cada ensayo sin tocar test.
train_raw_split_df = train_df.copy()
val_raw_split_df = val_df.copy()
test_raw_split_df = test_df.copy()

# =========================
# Pesos LDS para WSMOTER
# IMPORTANTE: se calculan solo con train_df para evitar fuga de información.
# Val/test NO tienen columna Gradient Weight.
# =========================
train_df = add_gradient_weights_lds(
    train_df,
    target_col=TARGET_COL,
    weight_col="Gradient Weight",
    sigma=LDS_SIGMA,
    n_bins=15
)

count_gt_1 = int(np.sum(train_df["Gradient Weight"].values > 1))
percent_gt_1 = 100 * count_gt_1 / len(train_df)
print(f"[LDS train] Valores > 1: {count_gt_1} de {len(train_df)} ({percent_gt_1:.1f}%)")
print("¿Gradient Weight en val?:", "Gradient Weight" in val_df.columns)
print("¿Gradient Weight en test?:", "Gradient Weight" in test_df.columns)

# =========================
# Escalado tabular
# Se ajusta solo con train original.
# Latitude y Longitude quedan normalizadas para el modelo.
# Latitude_raw y Longitude_raw permanecen sin normalizar para mapas.
# =========================
scaler_tab = StandardScaler()
train_df[scale_cols] = scaler_tab.fit_transform(train_df[scale_cols])
val_df[scale_cols]   = scaler_tab.transform(val_df[scale_cols])
test_df[scale_cols]  = scaler_tab.transform(test_df[scale_cols])

joblib.dump(scaler_tab, OUT_DIR / "tabular_scaler.pkl")
with open(OUT_DIR / "tabular_columns.json", "w", encoding="utf-8") as fh:
    json.dump(
        {
            "tabular_cols": tabular_cols,
            "scale_cols": scale_cols,
            "geo_lat_col": GEO_LAT_COL,
            "geo_lon_col": GEO_LON_COL
        },
        fh,
        ensure_ascii=False,
        indent=2
    )

# Guardar copia del train original para métricas e imagen stats.
# Esta copia NO incluye muestras sintéticas.
train_original_df = train_df.copy()

# ============================================================
# WSMOTER tabular
# ============================================================
def wsmoter_oversample_regression(
    df,
    feature_cols,
    target_col,
    id_col="ID",
    weight_col="Gradient Weight",
    multiplier=1.35,
    k_neighbors=5,
    weight_power=1.5,
    random_state=42,
    id_mode="anchor"
):
    """
    Implementación práctica tipo WSMOTER para regresión.

    Qué hace:
    - Genera nuevas filas tabulares sintéticas en train.
    - Selecciona muestras ancla con probabilidad proporcional a sus pesos LDS.
    - Interpola las variables tabulares indicadas en feature_cols y el target.
    - Mantiene el vínculo imagen-tabla asignando a cada fila sintética un ID real.

    Nota importante para este proyecto:
    - Latitude y Longitude entran al modelo, pero NO se interpolan en WSMOTER.
      La fila sintética conserva las coordenadas normalizadas y coordenadas raw
      del ID fuente para mantener coherencia con la imagen Landsat asociada.
    - Val/test no tienen Gradient Weight ni pasan por WSMOTER.
    """
    if multiplier <= 1.0:
        out = df.copy()
        out["is_synthetic"] = 0
        out["source_ID"] = out[id_col].astype(int)
        return out.reset_index(drop=True)

    rng = np.random.default_rng(random_state)

    base = df.reset_index(drop=True).copy()
    n = len(base)
    n_new = int(round((multiplier - 1.0) * n))

    if n_new <= 0 or n < 2:
        base["is_synthetic"] = 0
        base["source_ID"] = base[id_col].astype(int)
        return base.reset_index(drop=True)

    k = min(k_neighbors + 1, n)

    X = base[feature_cols].to_numpy(dtype=np.float32)
    y = base[target_col].to_numpy(dtype=np.float32)

    if weight_col in base.columns:
        w = base[weight_col].to_numpy(dtype=np.float64)
        w = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
        w = np.maximum(w, 1e-8)
    else:
        # Si no existe weight_col, se pondera por rareza simple.
        # Esta ruta no debería usarse en train porque los pesos LDS se calculan antes.
        y_center = np.median(y)
        mad = np.median(np.abs(y - y_center)) + 1e-8
        w = 1.0 + np.abs(y - y_center) / mad

    probs = np.power(w, weight_power)
    probs = probs / probs.sum()

    nn = NearestNeighbors(n_neighbors=k, metric="euclidean")
    nn.fit(X)
    neigh_idx = nn.kneighbors(X, return_distance=False)

    synthetic_rows = []

    for _ in range(n_new):
        i = int(rng.choice(np.arange(n), p=probs))

        candidates = neigh_idx[i]
        candidates = candidates[candidates != i]

        if candidates.size == 0:
            j = i
        else:
            cand_w = w[candidates]
            cand_p = cand_w / cand_w.sum()
            j = int(rng.choice(candidates, p=cand_p))

        lam = rng.uniform(0.0, 1.0)

        xi = X[i]
        xj = X[j]

        yi = y[i]
        yj = y[j]

        x_new = xi + lam * (xj - xi)
        y_new = yi + lam * (yj - yi)

        # Se parte de la fila ancla.
        # Por eso se conservan ID, Latitude, Longitude, Latitude_raw y Longitude_raw
        # coherentes con la imagen asociada.
        row = base.iloc[i].copy()

        for col, val in zip(feature_cols, x_new):
            row[col] = float(val)

        row[target_col] = float(y_new)

        if weight_col in base.columns:
            row[weight_col] = float(w[i] + lam * (w[j] - w[i]))

        if id_mode == "neighbor":
            source_id = int(base.iloc[j][id_col])
            # Si se usa neighbor, también se conservan las coordenadas del vecino.
            row[id_col] = source_id
            row["Latitude"] = base.iloc[j]["Latitude"]
            row["Longitude"] = base.iloc[j]["Longitude"]
            row[GEO_LAT_COL] = base.iloc[j][GEO_LAT_COL]
            row[GEO_LON_COL] = base.iloc[j][GEO_LON_COL]
        else:
            source_id = int(base.iloc[i][id_col])
            row[id_col] = source_id

        row["source_ID"] = source_id
        row["is_synthetic"] = 1

        synthetic_rows.append(row)

    synth_df = pd.DataFrame(synthetic_rows)

    base["is_synthetic"] = 0
    base["source_ID"] = base[id_col].astype(int)

    out = pd.concat([base, synth_df], axis=0, ignore_index=True)
    out = out.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    return out

# Columnas usadas para generar muestras sintéticas.
# Latitude y Longitude sí entran al modelo, pero se excluyen de la interpolación
# para mantener coherencia con la imagen real asociada por ID.
WSMOTER_FEATURE_COLS = [c for c in tabular_cols if c not in ["Latitude", "Longitude"]]

print("Columnas usadas por WSMOTER:", WSMOTER_FEATURE_COLS)
print("Columnas tabulares de entrada al modelo:", tabular_cols)

if USE_WSMOTER_TABULAR:
    train_df = wsmoter_oversample_regression(
        df=train_df,
        feature_cols=WSMOTER_FEATURE_COLS,
        target_col=TARGET_COL,
        id_col="ID",
        weight_col="Gradient Weight",
        multiplier=WSMOTER_MULTIPLIER,
        k_neighbors=WSMOTER_K_NEIGHBORS,
        weight_power=WSMOTER_WEIGHT_POWER,
        random_state=SEED,
        id_mode=WSMOTER_ID_MODE
    )
else:
    train_df["is_synthetic"] = 0
    train_df["source_ID"] = train_df["ID"].astype(int)

val_df["is_synthetic"] = 0
val_df["source_ID"] = val_df["ID"].astype(int)

test_df["is_synthetic"] = 0
test_df["source_ID"] = test_df["ID"].astype(int)

train_original_df["is_synthetic"] = 0
train_original_df["source_ID"] = train_original_df["ID"].astype(int)

# Seguridad: validación y test no deben tener pesos LDS.
val_df = val_df.drop(columns=["Gradient Weight"], errors="ignore")
test_df = test_df.drop(columns=["Gradient Weight"], errors="ignore")

print(
    f"Train después de WSMOTER: {len(train_df)} "
    f"(originales={int((train_df['is_synthetic'] == 0).sum())}, "
    f"sintéticas={int((train_df['is_synthetic'] == 1).sum())})"
)
print(f"Val: {len(val_df)} | Test: {len(test_df)}")
print("¿Gradient Weight en train?:", "Gradient Weight" in train_df.columns)
print("¿Gradient Weight en val?:", "Gradient Weight" in val_df.columns)
print("¿Gradient Weight en test?:", "Gradient Weight" in test_df.columns)

# =========================
# Transformación del target
# Se aplica después de WSMOTER para que las filas sintéticas entren al entrenamiento.
# =========================
for _df in [train_df, train_original_df, val_df, test_df]:
    _df[TARGET_COL + "_raw"] = _df[TARGET_COL].copy()

min_train_target = train_original_df[TARGET_COL].min()
target_shift = 0.0

if min_train_target <= -1:
    target_shift = float(abs(min_train_target) + 1.0)
    print(f"Aplicando shift al target: {target_shift:.6f}")

    for _df in [train_df, train_original_df, val_df, test_df]:
        _df[TARGET_COL] += target_shift

for _df in [train_df, train_original_df, val_df, test_df]:
    _df[TARGET_COL] = np.log1p(_df[TARGET_COL].astype(float))

scaler_y = StandardScaler()
train_df[[TARGET_COL]] = scaler_y.fit_transform(train_df[[TARGET_COL]])

# Mismo escalador para train original, val y test
train_original_df[[TARGET_COL]] = scaler_y.transform(train_original_df[[TARGET_COL]])
val_df[[TARGET_COL]] = scaler_y.transform(val_df[[TARGET_COL]])
test_df[[TARGET_COL]] = scaler_y.transform(test_df[[TARGET_COL]])

joblib.dump(scaler_y, OUT_DIR / "target_scaler.pkl")
joblib.dump({"shift": target_shift}, OUT_DIR / "target_shift.json")

# ============================================================
# Estadísticas de imagen
# Se calculan con train original, no con el train sintético,
# para no sesgar las estadísticas por IDs repetidos.
# ============================================================
image_mean, image_std = compute_image_stats_train(train_original_df)
thermal_mean, thermal_std = compute_thermal_stats_train(train_original_df)

joblib.dump(
    {"mean": image_mean, "std": image_std, "band_order": BAND_ORDER},
    OUT_DIR / "image_stats.pkl"
)

thermal_channel_names = [BAND_ORDER[i] for i in THERMAL_DERIVED_IDXS]

joblib.dump(
    {
        "mean": thermal_mean,
        "std": thermal_std,
        "channels": thermal_channel_names
    },
    OUT_DIR / "thermal_stats.pkl"
)

# Guardar trazabilidad de los splits
train_df.to_csv(OUT_DIR / "train_wsmoter.csv", index=False)
train_original_df.to_csv(OUT_DIR / "train_original_no_wsmoter.csv", index=False)
val_df.to_csv(OUT_DIR / "val.csv", index=False)
test_df.to_csv(OUT_DIR / "test.csv", index=False)

print("Image mean:", image_mean)
print("Image std:", image_std)
print("Thermal/derived channels:", thermal_channel_names)
print("Thermal/derived mean:", thermal_mean)
print("Thermal/derived std:", thermal_std)


In [ ]:
# Cell X: diagrama de pesos LDS por bins del gradiente geotérmico

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ==========================================================
# DataFrame a usar para el gráfico
# Recomendado: train_original_df, porque contiene los pesos
# LDS calculados sobre el train original, sin muestras sintéticas
# ==========================================================
plot_df = train_original_df.copy()

target_col = TARGET_COL
weight_col = "Gradient Weight"

# Si definiste un número fijo de bins, úsalo.
# Si no existe, se usan 15 bins (16 bordes)
n_bins_plot = LDS_N_BINS if "LDS_N_BINS" in globals() else 10

# Verificación básica
if weight_col not in plot_df.columns:
    raise ValueError(
        f"No existe la columna '{weight_col}' en plot_df. "
        "Asegúrate de haber ejecutado add_gradient_weights_lds(...) antes de esta celda."
    )

# ==========================================================
# Crear bordes de bins
# ==========================================================
bin_edges = np.linspace(
    plot_df[target_col].min(),
    plot_df[target_col].max(),
    num=n_bins_plot + 1
)

# Asignar cada valor a su bin correspondiente
plot_df["bin"] = np.digitize(
    plot_df[target_col],
    bins=bin_edges,
    right=True
)

# Corregir extremos para que siempre queden dentro de [1, n_bins_plot]
plot_df["bin"] = plot_df["bin"].clip(1, n_bins_plot)

# ==========================================================
# Peso promedio por bin
# ==========================================================
bin_to_weight = plot_df.groupby("bin")[weight_col].mean()

# Asegurar que todos los bins existan, aunque estén vacíos
all_bins = range(1, n_bins_plot + 1)
bin_to_weight_full = bin_to_weight.reindex(all_bins).fillna(0.0)

# ==========================================================
# Normalización de pesos a [0, 1]
# ==========================================================
w_min = bin_to_weight_full.min()
w_max = bin_to_weight_full.max()

if w_max > w_min:
    bin_weights_norm = (bin_to_weight_full - w_min) / (w_max - w_min)
else:
    bin_weights_norm = pd.Series(0.5, index=bin_to_weight_full.index)

# ==========================================================
# Figura
# ==========================================================
fig, ax = plt.subplots(figsize=(10, 6))

# Dibujar histograma
n, bins, patches = ax.hist(
    plot_df[target_col],
    bins=bin_edges,
    edgecolor="black"
)

# Colormap
cmap = plt.cm.jet
norm = mcolors.Normalize(
    vmin=bin_weights_norm.min(),
    vmax=bin_weights_norm.max()
)

# Colorear cada bin según su peso medio normalizado
for i, patch in enumerate(patches):
    bin_number = i + 1
    color = cmap(norm(bin_weights_norm.loc[bin_number]))
    patch.set_facecolor(color)

# ==========================================================
# Etiquetas
# ==========================================================
ax.set_xlabel("Gradiente Geotérmico Aparente (°C/Km)")
ax.set_ylabel("Frecuencia")
ax.set_title("Histograma del Gradiente Geotérmico Aparente coloreado por peso LDS")

# Barra de color
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array(bin_weights_norm.values)

cbar = fig.colorbar(
    sm,
    ax=ax,
    ticks=np.linspace(bin_weights_norm.min(), bin_weights_norm.max(), num=5)
)
cbar.ax.set_yticklabels(["Peso bajo", "", "", "", "Peso alto"])
cbar.set_label("Peso promedio normalizado por bin")

plt.tight_layout()
plt.show()

# Opcional: eliminar columna temporal
plot_df.drop(columns=["bin"], inplace=True, errors="ignore")

In [ ]:
# Cell X: histograma del target original coloreado por peso LDS real
# Usa TARGET_COL_raw para revisar los pesos calculados antes de la transformación del target

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plot_df = train_original_df.copy()

weight_col = "Gradient Weight"
target_raw_col = TARGET_COL + "_raw"

if weight_col not in plot_df.columns:
    raise ValueError(
        f"No existe '{weight_col}' en train_original_df. "
        "Ejecuta primero add_gradient_weights_lds(...)."
    )

if target_raw_col not in plot_df.columns:
    raise ValueError(
        f"No existe '{target_raw_col}'. "
        "Esta columna debe crearse antes de transformar el target."
    )

n_bins_plot = LDS_N_BINS if "LDS_N_BINS" in globals() else 15

# Bordes sobre el target original, no sobre el target escalado
bin_edges = np.linspace(
    plot_df[target_raw_col].min(),
    plot_df[target_raw_col].max(),
    num=n_bins_plot + 1
)

plot_df["bin"] = np.digitize(
    plot_df[target_raw_col],
    bins=bin_edges,
    right=True
)

plot_df["bin"] = plot_df["bin"].clip(1, n_bins_plot)

# Peso LDS promedio real por bin
bin_to_weight = plot_df.groupby("bin")[weight_col].mean()
all_bins = range(1, n_bins_plot + 1)
bin_to_weight_full = bin_to_weight.reindex(all_bins)

valid_weights = bin_to_weight_full.dropna()

if len(valid_weights) == 0:
    raise ValueError("No hay pesos válidos para graficar.")

w_min = valid_weights.min()
w_max = valid_weights.max()

if w_max == w_min:
    w_min -= 1e-6
    w_max += 1e-6

fig, ax = plt.subplots(figsize=(10, 6))

counts, bins, patches = ax.hist(
    plot_df[target_raw_col],
    bins=bin_edges,
    edgecolor="black"
)

cmap = plt.cm.jet
norm = mcolors.Normalize(vmin=w_min, vmax=w_max)

for i, patch in enumerate(patches):
    bin_number = i + 1
    weight_value = bin_to_weight_full.loc[bin_number]

    if pd.isna(weight_value):
        patch.set_facecolor("lightgray")
    else:
        patch.set_facecolor(cmap(norm(weight_value)))

ax.set_xlabel("Gradiente Geotérmico Aparente original (°C/km)")
ax.set_ylabel("Frecuencia")
ax.set_title("Histograma del Gradiente Geotérmico Aparente coloreado por peso LDS real")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array(valid_weights.values)

cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Peso LDS promedio real por bin")

plt.tight_layout()
plt.show()

In [ ]:
# Cell X: diagnóstico numérico de pesos LDS por bin

diagnostic_df = plot_df.groupby("bin").agg(
    y_min=(target_raw_col, "min"),
    y_max=(target_raw_col, "max"),
    frecuencia=(target_raw_col, "count"),
    peso_promedio=(weight_col, "mean"),
    peso_min=(weight_col, "min"),
    peso_max=(weight_col, "max")
).reset_index()

display(diagnostic_df)

In [ ]:
# Cell 12
# Ya no se descarga Prithvi.
# El backbone visual ahora es ConvNeXt-Tiny desde timm, adaptado a N_IMAGE_CHANNELS.

print("Backbone visual:", CONVNEXT_BACKBONE)
print("Pretrained:", PRETRAINED_CONVNEXT)
print("Canales ConvNeXt:", N_IMAGE_CHANNELS)
print("Canales rama térmica/derivados:", N_THERMAL_DERIVED_CHANNELS)
print("Bandas:", BAND_ORDER)
print("Rama térmica/derivados:", [BAND_ORDER[i] for i in THERMAL_DERIVED_IDXS])


In [ ]:
class MultibandPairAugment:
    def __init__(
        self,
        p_hflip=0.5,
        p_vflip=0.5,
        p_rot90=0.5,
        p_noise=0.35,
        noise_sigma_image=0.025,
        noise_sigma_thermal=0.020,
        clip_value=5.0
    ):
        self.p_hflip = p_hflip
        self.p_vflip = p_vflip
        self.p_rot90 = p_rot90
        self.p_noise = p_noise
        self.noise_sigma_image = noise_sigma_image
        self.noise_sigma_thermal = noise_sigma_thermal
        self.clip_value = clip_value

    def __call__(self, optical, thermal):
        """
        optical: tensor [C, H, W]
        thermal: tensor [C_aux, H, W]
        """
        # Flip horizontal
        if torch.rand(1).item() < self.p_hflip:
            optical = torch.flip(optical, dims=[2])
            thermal = torch.flip(thermal, dims=[2])

        # Flip vertical
        if torch.rand(1).item() < self.p_vflip:
            optical = torch.flip(optical, dims=[1])
            thermal = torch.flip(thermal, dims=[1])

        # Rotación en múltiplos de 90 grados
        if torch.rand(1).item() < self.p_rot90:
            k = torch.randint(0, 4, (1,)).item()
            optical = torch.rot90(optical, k=k, dims=[1, 2])
            thermal = torch.rot90(thermal, k=k, dims=[1, 2])

        # Ruido gaussiano suave, después de normalizar
        if torch.rand(1).item() < self.p_noise:
            optical = optical + torch.randn_like(optical) * self.noise_sigma_image
            thermal = thermal + torch.randn_like(thermal) * self.noise_sigma_thermal

            optical = torch.clamp(optical, -self.clip_value, self.clip_value)
            thermal = torch.clamp(thermal, -self.clip_value, self.clip_value)

        return optical, thermal


class MultimodalDataset(Dataset):
    def __init__(self, df, train=False, augment=None):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.augment = augment

        image_stats = joblib.load(OUT_DIR / "image_stats.pkl")
        self.img_mean = np.asarray(image_stats["mean"], dtype=np.float32)
        self.img_std  = np.asarray(image_stats["std"], dtype=np.float32)

        thermal_stats = joblib.load(OUT_DIR / "thermal_stats.pkl")
        self.th_mean = np.asarray(thermal_stats["mean"], dtype=np.float32)
        self.th_std  = np.asarray(thermal_stats["std"], dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # En filas sintéticas WSMOTER, ID corresponde a una imagen real.
        # source_ID se guarda solo como trazabilidad.
        arr, _ = read_geotiff(BASE_DIR / f"LANDSAT7_{int(row.ID)}.tif")

        # optical = stack completo de 9/10 canales
        # thermal = Band61 + NDVI + Band3/Band1 + Band5/Band7
        optical, thermal = split_optical_thermal(arr)

        # Relleno de faltantes
        optical = fill_missing_median(optical)
        thermal = fill_missing_median(thermal)
        #optical = fill_missing_knn_mean(optical)
        #thermal = fill_missing_knn_mean(thermal)

        # Resize
        optical = resize_multiband(optical, TARGET_SIZE)
        thermal = resize_multiband(thermal, TARGET_SIZE)

        # Normalización por canal para ConvNeXt
        optical = optical.astype(np.float32)

        for b in range(optical.shape[0]):
            optical[b] = (optical[b] - self.img_mean[b]) / (self.img_std[b] + 1e-8)

        optical = np.clip(optical, -5.0, 5.0)

        # Normalización por canal para rama térmica/derivados
        thermal = thermal.astype(np.float32)

        for b in range(thermal.shape[0]):
            thermal[b] = (thermal[b] - self.th_mean[b]) / (self.th_std[b] + 1e-8)

        thermal = np.clip(thermal, -5.0, 5.0)

        optical = torch.from_numpy(optical.astype(np.float32))   # [C, H, W]
        thermal = torch.from_numpy(thermal.astype(np.float32))   # [4, H, W]

        # Las aumentaciones espaciales y el ruido suave se aplican SOLO
        # a las muestras sintéticas generadas por WSMOTER.
        is_synthetic = int(row["is_synthetic"]) if "is_synthetic" in row.index else 0

        if self.train and self.augment is not None and is_synthetic == 1:
            optical, thermal = self.augment(optical, thermal)

        tab_vals = row[tabular_cols].to_numpy(dtype=np.float32, copy=True)
        tab = torch.from_numpy(tab_vals)

        y = torch.tensor(np.float32(row[TARGET_COL]), dtype=torch.float32)

        return {
            "optical": optical,
            "thermal": thermal,
            "tabular": tab,
            "target": y,
            "id": int(row.ID),
            "source_id": int(row.source_ID) if "source_ID" in row.index else int(row.ID),
            "is_synthetic": is_synthetic
        }


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

def _valid_num_groups(channels, preferred=(16, 8, 4, 2, 1)):
    for g in preferred:
        if g <= channels and channels % g == 0:
            return g
    return 1

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2)

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        attn = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attn

class SEVector(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(1, channels // reduction)

        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        s = self.fc(x)
        return x * s

class SqueezeExcite2D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(8, channels // reduction)

        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.GELU(),
            nn.Linear(mid, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        s = F.adaptive_avg_pool2d(x, 1).flatten(1)
        s = self.fc(s).unsqueeze(-1).unsqueeze(-1)
        return x * s

class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size,
                stride=stride,
                padding=padding,
                bias=False
            ),
            nn.GroupNorm(_valid_num_groups(out_ch), out_ch),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)

class ThermalResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, drop=0.0):
        super().__init__()

        self.conv1 = ConvGNAct(
            in_ch,
            out_ch,
            kernel_size=3,
            stride=stride,
            padding=1
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(
                out_ch,
                out_ch,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.GroupNorm(_valid_num_groups(out_ch), out_ch)
        )

        self.se = SqueezeExcite2D(out_ch, reduction=8)
        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()

        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(
                    in_ch,
                    out_ch,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.GroupNorm(_valid_num_groups(out_ch), out_ch)
            )
        else:
            self.skip = nn.Identity()

        self.act = nn.GELU()

    def forward(self, x):
        identity = self.skip(x)

        out = self.conv1(x)
        out = self.conv2(out)
        out = self.se(out)
        out = self.drop(out)

        out = out + identity
        return self.act(out)

class MultiScaleTokenize(nn.Module):
    def __init__(self, in_channels_list, d_model=512, token_pool_size=(4, 4)):
        super().__init__()

        assert len(in_channels_list) > 0

        self.token_pool_size = token_pool_size
        self.projs = nn.ModuleList([
            nn.Conv2d(in_ch, d_model, kernel_size=1)
            for in_ch in in_channels_list
        ])

    def forward(self, feats):
        tokens_list = []

        for f, proj in zip(feats, self.projs):
            x = proj(f)
            x = F.adaptive_avg_pool2d(x, self.token_pool_size)

            B, C, H, W = x.shape

            t = x.view(B, C, H * W).permute(0, 2, 1)
            tokens_list.append(t)

        tokens = torch.cat(tokens_list, dim=1)
        return tokens

In [ ]:
class ConvNeXt9_Multimodal_BiAttn_Residual(nn.Module):
    def __init__(
        self,
        n_tabular,
        n_image_channels=9,
        n_thermal_channels=4,
        backbone_name="convnext_tiny",
        pretrained=True,
        d_model=512,
        n_heads=8,
        dropout=0.25,
        token_pool_size=(4, 4),
        img_token_layers=2,
        thermal_token_layers=1,
        img_dim_feedforward=2048,
        thermal_dim_feedforward=None,
        tab_hidden=256,
        head_hidden_1=768,
        head_hidden_2=256,
        se_fusion_reduction=8,
        spatial_attention_kernel=7
    ):
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError(f"d_model={d_model} debe ser divisible por n_heads={n_heads}")

        if thermal_dim_feedforward is None:
            thermal_dim_feedforward = 4 * d_model

        # Rama visual principal: ConvNeXt-Tiny con 9 canales
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            in_chans=n_image_channels,
            features_only=True,
            out_indices=(0, 1, 2, 3)
        )

        img_channels = self.backbone.feature_info.channels()
        last_ch = img_channels[-1]

        self.spatial_attn = SpatialAttention(kernel_size=spatial_attention_kernel)

        self.multiscale_tokenize = MultiScaleTokenize(
            in_channels_list=img_channels,
            d_model=d_model,
            token_pool_size=token_pool_size
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=img_dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.img_token_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=img_token_layers
        )

        self.img_vec_proj = nn.Sequential(
            nn.Linear(last_ch, d_model),
            nn.GELU()
        )

        self.img_token_ln = nn.LayerNorm(d_model)
        self.img_vec_ln = nn.LayerNorm(d_model)

        # Rama tabular
        self.tab_mlp = nn.Sequential(
            nn.Linear(n_tabular, tab_hidden),
            nn.LayerNorm(tab_hidden),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(tab_hidden, tab_hidden),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(tab_hidden, d_model),
            nn.GELU()
        )

        self.tab_ln = nn.LayerNorm(d_model)

        # Rama térmica + derivados
        self.thermal_stem = ConvGNAct(
            n_thermal_channels,
            32,
            kernel_size=3,
            stride=2,
            padding=1
        )

        self.thermal_stage1 = ThermalResBlock(32, 64, stride=2, drop=0.05)
        self.thermal_stage2 = ThermalResBlock(64, 128, stride=2, drop=0.05)
        self.thermal_stage3 = ThermalResBlock(128, 256, stride=2, drop=0.10)

        self.thermal_context = nn.Sequential(
            ThermalResBlock(256, 256, stride=1, drop=0.10),
            ThermalResBlock(256, 256, stride=1, drop=0.10)
        )

        self.thermal_token_projs = nn.ModuleList([
            nn.Sequential(nn.Linear(32 * 2, d_model), nn.GELU()),
            nn.Sequential(nn.Linear(64 * 2, d_model), nn.GELU()),
            nn.Sequential(nn.Linear(128 * 2, d_model), nn.GELU()),
            nn.Sequential(nn.Linear(256 * 2, d_model), nn.GELU()),
        ])

        thermal_encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=thermal_dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.thermal_token_encoder = nn.TransformerEncoder(
            thermal_encoder_layer,
            num_layers=thermal_token_layers
        )

        self.thermal_ln = nn.LayerNorm(d_model)

        # Atención cruzada imagen <-> tabla
        self.attn_img_to_tab = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            batch_first=True,
            dropout=dropout
        )

        self.attn_tab_to_img = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            batch_first=True,
            dropout=dropout
        )

        # fusión por concatenación + proyección
        self.img_cross_fuse = nn.Sequential(
            nn.LayerNorm(d_model * 2),
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model)
        )

        self.tab_cross_fuse = nn.Sequential(
            nn.LayerNorm(d_model * 2),
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model)
        )


        self.concat_gate = nn.Sequential(
            nn.LayerNorm(d_model * 3),
            nn.Linear(d_model * 3, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model * 3)
        )

        self.se_fusion = SEVector(d_model * 3, reduction=se_fusion_reduction)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model * 3),
            nn.Linear(d_model * 3, head_hidden_1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(head_hidden_1, head_hidden_2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),

            nn.Linear(head_hidden_2, 1)
        )

        self._init_new_weights()

    def _init_new_weights(self):
        """
        Inicializa solo los módulos nuevos.
        No reinicializa ConvNeXt preentrenado.
        """
        modules_to_init = [
            self.spatial_attn,
            self.multiscale_tokenize,
            self.img_token_encoder,
            self.img_vec_proj,
            self.img_token_ln,
            self.img_vec_ln,

            self.tab_mlp,
            self.tab_ln,

            self.thermal_stem,
            self.thermal_stage1,
            self.thermal_stage2,
            self.thermal_stage3,
            self.thermal_context,
            self.thermal_token_projs,
            self.thermal_token_encoder,
            self.thermal_ln,

            self.attn_img_to_tab,
            self.attn_tab_to_img,

            # Nuevos bloques
            self.img_cross_fuse,
            self.tab_cross_fuse,
            self.concat_gate,

            self.se_fusion,
            self.head
        ]

        for module in modules_to_init:
            for m in module.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

                elif isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(
                        m.weight,
                        mode="fan_out",
                        nonlinearity="relu"
                    )
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

    def _image_forward(self, img):
        """
        img: [B, 9, H, W]
        """
        feats = self.backbone(img)

        f1, f2, f3, f4 = feats

        f4_att = self.spatial_attn(f4)

        tokens = self.multiscale_tokenize([f1, f2, f3, f4_att])
        tokens = self.img_token_encoder(tokens)
        tokens = self.img_token_ln(tokens)

        z_spatial = F.adaptive_avg_pool2d(f4_att, 1).flatten(1)
        z_img = self.img_vec_proj(z_spatial)
        z_img = self.img_vec_ln(z_img)

        return z_img, tokens

    def _thermal_forward(self, thermal):
        """
        thermal: [B, 4, H, W]
        """
        t1 = self.thermal_stem(thermal)
        t2 = self.thermal_stage1(t1)
        t3 = self.thermal_stage2(t2)
        t4 = self.thermal_stage3(t3)
        t4 = self.thermal_context(t4)

        thermal_feats = [t1, t2, t3, t4]
        thermal_tokens = []

        for f, proj in zip(thermal_feats, self.thermal_token_projs):
            th_avg = F.adaptive_avg_pool2d(f, 1).flatten(1)
            th_max = F.adaptive_max_pool2d(f, 1).flatten(1)

            token = proj(torch.cat([th_avg, th_max], dim=1)).unsqueeze(1)
            thermal_tokens.append(token)

        thermal_tokens = torch.cat(thermal_tokens, dim=1)
        thermal_tokens = self.thermal_token_encoder(thermal_tokens)

        z_th = thermal_tokens.mean(dim=1)
        z_th = self.thermal_ln(z_th)

        return z_th

    def forward(self, img, thermal, tab):
        # Rama imagen ConvNeXt 9 canales
        z_img, img_tokens = self._image_forward(img)

        # Rama tabular
        z_tab = self.tab_mlp(tab)
        z_tab = self.tab_ln(z_tab)

        # Atención bidireccional imagen - tabla
        tab_tokens = z_tab.unsqueeze(1)

        img_attn_out, _ = self.attn_img_to_tab(
            query=img_tokens,
            key=tab_tokens,
            value=tab_tokens
        )

        z_img_att = img_attn_out.mean(dim=1)

        tab_attn_out, _ = self.attn_tab_to_img(
            query=tab_tokens,
            key=img_tokens,
            value=img_tokens
        )

        z_tab_att = tab_attn_out.squeeze(1)

        # concatenación + proyección
        # Cada salida vuelve a quedar en d_model.
        z_img_final = self.img_cross_fuse(
            torch.cat([z_img, z_img_att], dim=1)
        )

        z_tab_final = self.tab_cross_fuse(
            torch.cat([z_tab, z_tab_att], dim=1)
        )

        # Rama térmica + derivados
        z_th = self._thermal_forward(thermal)

        # Gates + SE + regresión
        fusion_pre = torch.cat(
            [z_img_final, z_tab_final, z_th],
            dim=1
        )

        gate_raw = self.concat_gate(fusion_pre)
        g_img_raw, g_tab_raw, g_th_raw = gate_raw.chunk(3, dim=1)

        scale_img = 1.0 + 0.5 * torch.tanh(g_img_raw)
        scale_tab = 1.0 + 0.5 * torch.tanh(g_tab_raw)
        scale_th = 1.0 + 0.5 * torch.tanh(g_th_raw)

        fusion = torch.cat(
            [
                z_img_final * scale_img,
                z_tab_final * scale_tab,
                z_th * scale_th
            ],
            dim=1
        )

        fusion = self.se_fusion(fusion)

        out = self.head(fusion).squeeze(1)

        return out

In [ ]:
# HybridWeightedSampler eliminado.
# El balanceo del conjunto de entrenamiento ahora se hace antes de construir el Dataset,
# mediante WSMOTER tabular en train_df.
# Las imágenes no se sintetizan: cada fila conserva un ID real que apunta a un GeoTIFF real.
print("HybridWeightedSampler eliminado. Oversampling activo en train_df mediante WSMOTER:", USE_WSMOTER_TABULAR)


In [ ]:
target_scaler = joblib.load(OUT_DIR / "target_scaler.pkl")
target_shift = joblib.load(OUT_DIR / "target_shift.json")["shift"]

train_augment = MultibandPairAugment(
    p_hflip=AUG_P_HFLIP,
    p_vflip=AUG_P_VFLIP,
    p_rot90=AUG_P_ROT90,
    p_noise=AUG_P_NOISE,
    noise_sigma_image=AUG_NOISE_SIGMA_IMAGE,
    noise_sigma_thermal=AUG_NOISE_SIGMA_THERMAL,
    clip_value=AUG_CLIP_VALUE
) if USE_IMAGE_AUGMENTATION else None

train_dataset = MultimodalDataset(
    train_df,
    train=True,
    augment=train_augment
)

# Para métricas de train se usa el train original, sin filas sintéticas WSMOTER.
train_eval_dataset = MultimodalDataset(
    train_original_df,
    train=False,
    augment=None
)

val_dataset = MultimodalDataset(
    val_df,
    train=False,
    augment=None
)

test_dataset = MultimodalDataset(
    test_df,
    train=False,
    augment=None
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("Tamaño train_dataset:", len(train_dataset))
print("Tamaño train_eval_dataset original:", len(train_eval_dataset))
print("Tamaño val_dataset:", len(val_dataset))
print("Tamaño test_dataset:", len(test_dataset))

model = ConvNeXt9_Multimodal_BiAttn_Residual(
    n_tabular=len(tabular_cols),
    n_image_channels=N_IMAGE_CHANNELS,
    n_thermal_channels=N_THERMAL_DERIVED_CHANNELS,
    backbone_name=CONVNEXT_BACKBONE,
    pretrained=PRETRAINED_CONVNEXT,
    d_model=512,
    n_heads=8,
    dropout=0.25,
    token_pool_size=(4, 4)
).to(DEVICE)

criterion = torch.nn.HuberLoss(delta=0.5)

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

def get_new_modules(model):
    """
    Módulos nuevos, excluyendo ConvNeXt preentrenado.
    """
    return [
        model.spatial_attn,
        model.multiscale_tokenize,
        model.img_token_encoder,
        model.img_vec_proj,
        model.img_token_ln,
        model.img_vec_ln,
        model.tab_mlp,
        model.tab_ln,
        model.thermal_stem,
        model.thermal_stage1,
        model.thermal_stage2,
        model.thermal_stage3,
        model.thermal_context,
        model.thermal_token_projs,
        model.thermal_token_encoder,
        model.thermal_ln,
        model.attn_img_to_tab,
        model.attn_tab_to_img,
        model.concat_gate,
        model.se_fusion,
        model.head
    ]

def unfreeze_modules(modules):
    for module in modules:
        for p in module.parameters():
            p.requires_grad = True

def collect_params(modules):
    params = []
    seen = set()

    for module in modules:
        for p in module.parameters():
            if p.requires_grad and id(p) not in seen:
                params.append(p)
                seen.add(id(p))

    return params

def configure_stage(model, stage="head"):
    freeze_all(model)

    if stage == "head":
        # Entrena solo módulos nuevos; ConvNeXt congelado.
        unfreeze_modules(get_new_modules(model))

    elif stage == "backbone":
        # Fine-tuning suave de ConvNeXt + módulos nuevos.
        unfreeze_modules(get_new_modules(model))
        unfreeze_modules([model.backbone])

    elif stage == "full":
        # Fine-tuning completo.
        for p in model.parameters():
            p.requires_grad = True

    else:
        raise ValueError(f"Stage no soportado: {stage}")

def build_optimizer(model, stage="head", hparams=None):
    hparams = hparams or {}
    lr_head = hparams.get("lr_head", LR_HEAD)
    lr_fine = hparams.get("lr_fine", LR_FINE)
    lr_full = hparams.get("lr_full", LR_FULL)
    weight_decay = hparams.get("weight_decay", WEIGHT_DECAY)

    new_modules = get_new_modules(model)

    if stage == "head":
        return torch.optim.AdamW(
            collect_params(new_modules),
            lr=lr_head,
            weight_decay=weight_decay
        )

    elif stage == "backbone":
        return torch.optim.AdamW(
            [
                {"params": collect_params([model.backbone]), "lr": lr_fine},
                {"params": collect_params(new_modules), "lr": lr_fine * 5.0},
            ],
            weight_decay=weight_decay
        )

    elif stage == "full":
        return torch.optim.AdamW(
            [
                {"params": collect_params([model.backbone]), "lr": lr_full},
                {"params": collect_params(new_modules), "lr": lr_full * 5.0},
            ],
            weight_decay=weight_decay
        )

    else:
        raise ValueError(f"Stage no soportado: {stage}")

def build_scheduler(optimizer, hparams=None):
    hparams = hparams or {}
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=hparams.get("scheduler_factor", 0.5),
        patience=hparams.get("scheduler_patience", 2),
        min_lr=hparams.get("scheduler_min_lr", 1e-6)
    )


In [ ]:
sample = MultimodalDataset(train_df, train=True, augment=train_augment)[0]

print("Optical shape:", sample["optical"].shape)
print("Thermal shape:", sample["thermal"].shape)
print("Tabular shape:", sample["tabular"].shape)
print("Target:", sample["target"])
print("ID:", sample["id"])
print("Source ID:", sample["source_id"])
print("Is synthetic:", sample["is_synthetic"])


In [ ]:
def inverse_target_transform(y_scaled, scaler, shift):
    y_scaled = np.array(y_scaled).reshape(-1, 1)
    y_log = scaler.inverse_transform(y_scaled).ravel()
    y = np.expm1(y_log)
    if shift != 0:
        y = y - shift
    return y

def safe_r2(y_true, y_pred):
    try:
        return r2_score(y_true, y_pred)
    except Exception:
        return float("nan")

def compute_real_metrics(preds_scaled, targs_scaled):
    preds_real = inverse_target_transform(preds_scaled, target_scaler, target_shift)
    targs_real = inverse_target_transform(targs_scaled, target_scaler, target_shift)

    rmse_real = np.sqrt(np.mean((targs_real - preds_real) ** 2))
    mae_real = mean_absolute_error(targs_real, preds_real)
    r2_real = safe_r2(targs_real, preds_real)

    return {
        "preds_real": preds_real,
        "targets_real": targs_real,
        "rmse_real": rmse_real,
        "mae_real": mae_real,
        "r2_real": r2_real
    }

def evaluate_loader(model, loader, criterion):
    model.eval()
    preds_list, targs_list, ids_list, losses = [], [], [], []

    with torch.no_grad():
        for batch in loader:
            optical = batch["optical"].to(DEVICE)
            thermal = batch["thermal"].to(DEVICE)
            tabular = batch["tabular"].to(DEVICE).float()
            targets = batch["target"].to(DEVICE).float()

            preds = model(optical, thermal, tabular)
            loss = criterion(preds, targets)

            losses.append(loss.item())
            preds_list.append(preds.cpu().numpy())
            targs_list.append(targets.cpu().numpy())
            ids_list.extend(batch["id"])

    preds_arr = np.concatenate(preds_list) if len(preds_list) > 0 else np.array([])
    targs_arr = np.concatenate(targs_list) if len(targs_list) > 0 else np.array([])

    if len(preds_arr) == 0:
        return {
            "loss": float("nan"),
            "rmse_scaled": float("nan"),
            "mae_scaled": float("nan"),
            "r2_scaled": float("nan"),
            "rmse_real": float("nan"),
            "mae_real": float("nan"),
            "r2_real": float("nan"),
            "preds": preds_arr,
            "targets": targs_arr,
            "preds_real": np.array([]),
            "targets_real": np.array([]),
            "ids": ids_list
        }

    rmse_scaled = np.sqrt(np.mean((targs_arr - preds_arr) ** 2))
    mae_scaled = mean_absolute_error(targs_arr, preds_arr)
    r2_scaled = safe_r2(targs_arr, preds_arr)

    real_metrics = compute_real_metrics(preds_arr, targs_arr)

    return {
        "loss": float(np.mean(losses)),
        "rmse_scaled": rmse_scaled,
        "mae_scaled": mae_scaled,
        "r2_scaled": r2_scaled,
        "rmse_real": real_metrics["rmse_real"],
        "mae_real": real_metrics["mae_real"],
        "r2_real": real_metrics["r2_real"],
        "preds": preds_arr,
        "targets": targs_arr,
        "preds_real": real_metrics["preds_real"],
        "targets_real": real_metrics["targets_real"],
        "ids": ids_list
    }

def get_current_lrs(optimizer):
    return [pg["lr"] for pg in optimizer.param_groups]

def train_phase(model, train_loader, train_eval_loader, val_loader, criterion,
                epochs, stage_name, ckpt_name, hparams=None, ckpt_dir=None,
                trial=None, step_offset=0):
    hparams = hparams or {}
    ckpt_dir = Path(ckpt_dir) if ckpt_dir is not None else OUT_DIR
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    configure_stage(model, stage_name)
    optimizer = build_optimizer(model, stage_name, hparams=hparams)
    scheduler = build_scheduler(optimizer, hparams=hparams)

    max_grad_norm = hparams.get("max_grad_norm", MAX_GRAD_NORM)
    best_val_rmse_real = np.inf
    history = []

    for ep in range(1, epochs + 1):
        model.train()
        train_losses = []
        pbar = tqdm(train_loader, desc=f"{stage_name.capitalize()} Epoch {ep}", leave=False,dynamic_ncols=True, mininterval=0.5, position=0)

        for batch in pbar:
            optical = batch["optical"].to(DEVICE)
            thermal = batch["thermal"].to(DEVICE)
            tabular = batch["tabular"].to(DEVICE).float()
            targets = batch["target"].to(DEVICE).float()

            optimizer.zero_grad()
            preds = model(optical, thermal, tabular)
            loss = criterion(preds, targets)
            loss.backward()

            trainable_params = [p for p in model.parameters() if p.requires_grad]
            if len(trainable_params) > 0 and max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)

            optimizer.step()

            train_losses.append(loss.item())
            pbar.set_postfix({"loss": f"{np.mean(train_losses):.4f}"}, refresh=True)

        train_metrics = evaluate_loader(model, train_eval_loader, criterion)
        val_metrics = evaluate_loader(model, val_loader, criterion)

        # El scheduler ve la pérdida de validación
        scheduler.step(val_metrics["loss"])

        current_lrs = ", ".join([f"{lr:.2e}" for lr in get_current_lrs(optimizer)])

        print(
            f"[{stage_name.upper()}] Epoch {ep}/{epochs} | "
            f"LR={current_lrs} | "
            f"TrainLoss={np.mean(train_losses):.4f} | "
            f"TrainRMSE_real={train_metrics['rmse_real']:.4f} | "
            f"TrainMAE_real={train_metrics['mae_real']:.4f} | "
            f"TrainR2_real={train_metrics['r2_real']:.4f}"
        )
        print(
            f"                 ValLoss={val_metrics['loss']:.4f} | "
            f"ValRMSE_real={val_metrics['rmse_real']:.4f} | "
            f"ValMAE_real={val_metrics['mae_real']:.4f} | "
            f"ValR2_real={val_metrics['r2_real']:.4f}"
        )

        history.append({
            "epoch": ep,
            "stage": stage_name,
            "train_loss": float(np.mean(train_losses)),
            "train_rmse_real": train_metrics["rmse_real"],
            "train_mae_real": train_metrics["mae_real"],
            "train_r2_real": train_metrics["r2_real"],
            "val_loss": val_metrics["loss"],
            "val_rmse_real": val_metrics["rmse_real"],
            "val_mae_real": val_metrics["mae_real"],
            "val_r2_real": val_metrics["r2_real"],
        })

        step = step_offset + ep
        if trial is not None:
            trial.report(val_metrics["rmse_real"], step=step)
            if trial.should_prune():
                raise optuna.TrialPruned(
                    f"Trial podado en {stage_name} epoch {ep}: "
                    f"ValRMSE_real={val_metrics['rmse_real']:.4f}"
                )

        # Guardar por métrica real de validación
        if val_metrics["rmse_real"] < best_val_rmse_real - 1e-6:
            best_val_rmse_real = val_metrics["rmse_real"]
            torch.save(
                {
                    "epoch": ep,
                    "stage": stage_name,
                    "model": model.state_dict(),
                    "opt": optimizer.state_dict(),
                    "val_loss": val_metrics["loss"],
                    "val_rmse_real": val_metrics["rmse_real"],
                    "val_mae_real": val_metrics["mae_real"],
                    "val_r2_real": val_metrics["r2_real"],
                    "hparams": hparams,
                },
                ckpt_dir / ckpt_name
            )
            print(f"  -> Mejor modelo ({stage_name}) guardado por ValRMSE_real.")

    return history


In [ ]:
## Cell 9: optimización de hiperparámetros con Optuna
def sample_hparams(trial):
    d_model = trial.suggest_categorical("d_model", [256, 384, 512])

    # n_heads condicionado a d_model para garantizar divisibilidad.
    if d_model == 384:
        n_heads = trial.suggest_categorical("n_heads_d384", [4, 6, 8])
    elif d_model == 256:
        n_heads = trial.suggest_categorical("n_heads_d256", [4, 8])
    else:
        n_heads = trial.suggest_categorical("n_heads_d512", [4, 8])

    # Cabeza regresora condicionada a d_model.
    if d_model == 256:
        head_hidden_1 = trial.suggest_categorical("head_hidden_1_d256", [384, 512, 768])
        head_hidden_2 = trial.suggest_categorical("head_hidden_2_d256", [128, 256, 384])
    elif d_model == 384:
        head_hidden_1 = trial.suggest_categorical("head_hidden_1_d384", [512, 768, 1024])
        head_hidden_2 = trial.suggest_categorical("head_hidden_2_d384", [192, 256, 384])
    else:
        head_hidden_1 = trial.suggest_categorical("head_hidden_1_d512", [768, 1024, 1536])
        head_hidden_2 = trial.suggest_categorical("head_hidden_2_d512", [256, 384, 512])

    token_pool_str = trial.suggest_categorical(
        "token_pool_size",
        ["2x2", "3x3", "4x4"]
    )
    token_pool_size = tuple(int(v) for v in token_pool_str.split("x"))

    img_ff_mult = trial.suggest_categorical("img_ff_mult", [2, 3, 4])
    thermal_ff_mult = trial.suggest_categorical("thermal_ff_mult", [2, 4])

    hparams = {
        # Arquitectura
        "d_model": d_model,
        "n_heads": n_heads,

        "dropout": trial.suggest_categorical(
            "dropout",
            [0.25, 0.30, 0.35, 0.40, 0.45]
        ),

        "token_pool_size": token_pool_size,

        "img_token_layers": trial.suggest_categorical(
            "img_token_layers",
            [1, 2, 3]
        ),

        "thermal_token_layers": trial.suggest_categorical(
            "thermal_token_layers",
            [1, 2]
        ),

        "img_dim_feedforward": img_ff_mult * d_model,
        "thermal_dim_feedforward": thermal_ff_mult * d_model,

        "tab_hidden": trial.suggest_categorical(
            "tab_hidden",
            [128, 256, 512]
        ),

        "head_hidden_1": head_hidden_1,
        "head_hidden_2": head_hidden_2,

        "se_fusion_reduction": trial.suggest_categorical(
            "se_fusion_reduction",
            [4, 8, 16]
        ),

        "spatial_attention_kernel": trial.suggest_categorical(
            "spatial_attention_kernel",
            [3, 5, 7]
        ),

        # Optimización
        "lr_head": trial.suggest_categorical(
            "lr_head",
            [5e-5, 1e-4, 2e-4, 3e-4, 5e-4]
        ),

        "lr_fine": trial.suggest_categorical(
            "lr_fine",
            [1e-6, 3e-6, 5e-6, 1e-5, 2e-5, 5e-5]
        ),

        "lr_full": trial.suggest_categorical(
            "lr_full",
            [1e-7, 3e-7, 5e-7, 1e-6, 3e-6]
        ),

        "weight_decay": trial.suggest_categorical(
            "weight_decay",
            [1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3]
        ),

        "max_grad_norm": trial.suggest_categorical(
            "max_grad_norm",
            [0.5, 1.0, 2.0, 3.0, 5.0]
        ),

        "huber_delta": trial.suggest_categorical(
            "huber_delta",
            [0.12, 0.25, 0.50, 0.75, 1.00, 1.50]
        ),

        "scheduler_factor": trial.suggest_categorical(
            "scheduler_factor",
            [0.3, 0.5, 0.7]
        ),

        "scheduler_patience": trial.suggest_categorical(
            "scheduler_patience",
            [1, 2, 3, 4]
        ),

        "scheduler_min_lr": trial.suggest_categorical(
            "scheduler_min_lr",
            [1e-8, 3e-8, 1e-7, 3e-7, 1e-6]
        ),

        # Duración fija del entrenamiento por fases
        "epochs_head": EPOCHS_HEAD,
        "epochs_backbone": EPOCHS_BACKBONE,
        "epochs_full": EPOCHS_FULL,

        # Datos / balanceo
        "batch_size": trial.suggest_categorical(
            "batch_size",
            [4, 8, 12, 16]
        ),

        "wsmoter_multiplier": trial.suggest_categorical(
            "wsmoter_multiplier",
            [1.1, 1.2, 1.3, 1.4]
        ),

        "wsmoter_weight_power": trial.suggest_categorical(
            "wsmoter_weight_power",
            [1.0, 1.5, 2.0, 2.5, 3.0]
        ),

        "lds_sigma": trial.suggest_categorical(
            "lds_sigma",
            [3.0, 5.0, 7.0, 10.0, 12.0]
        ),

        # ==========================================================
        # Aumentación
        # ==========================================================
        "use_image_augmentation": trial.suggest_categorical(
            "use_image_augmentation",
            [True, False]
        ),

        "aug_p_hflip": trial.suggest_categorical(
            "aug_p_hflip",
            [0.0, 0.25, 0.50, 0.60]
        ),

        "aug_p_vflip": trial.suggest_categorical(
            "aug_p_vflip",
            [0.0, 0.25, 0.50, 0.60]
        ),

        "aug_p_rot90": trial.suggest_categorical(
            "aug_p_rot90",
            [0.0, 0.25, 0.50, 0.60]
        ),

        "aug_p_noise": trial.suggest_categorical(
            "aug_p_noise",
            [0.0, 0.20, 0.35, 0.50, 0.60]
        ),

        "aug_noise_sigma_image": trial.suggest_categorical(
            "aug_noise_sigma_image",
            [0.005, 0.010, 0.020, 0.025, 0.035, 0.050]
        ),

        "aug_noise_sigma_thermal": trial.suggest_categorical(
            "aug_noise_sigma_thermal",
            [0.005, 0.010, 0.020, 0.025, 0.035, 0.050]
        ),
    }

    if not (hparams["lr_full"] <= hparams["lr_fine"] <= hparams["lr_head"]):
        raise optuna.TrialPruned(
            "Se poda porque no cumple lr_full <= lr_fine <= lr_head"
        )

    return hparams


def _jsonable_hparams(hparams):
    out = dict(hparams)
    if isinstance(out.get("token_pool_size"), tuple):
        out["token_pool_size"] = list(out["token_pool_size"])
    return out


def prepare_dataframes_for_hparams(hparams, include_test=False, persist=False):
    """
    Reconstruye los dataframes desde los splits crudos.
    En Optuna se llama con include_test=False para no tocar test dentro del objective.
    """
    global target_scaler, target_shift
    global train_df, train_original_df, val_df, test_df

    train_base = train_raw_split_df.reset_index(drop=True).copy()
    val_base = val_raw_split_df.reset_index(drop=True).copy()
    test_base = test_raw_split_df.reset_index(drop=True).copy() if include_test else None

    # Pesos LDS solo en train.
    train_base = add_gradient_weights_lds(
        train_base,
        target_col=TARGET_COL,
        weight_col="Gradient Weight",
        sigma=hparams["lds_sigma"]
    )

    # Escalado tabular ajustado solo con train original.
    scaler_tab_local = StandardScaler()
    train_base[scale_cols] = scaler_tab_local.fit_transform(train_base[scale_cols])
    val_base[scale_cols] = scaler_tab_local.transform(val_base[scale_cols])
    if include_test:
        test_base[scale_cols] = scaler_tab_local.transform(test_base[scale_cols])

    train_original_local = train_base.copy()

    if USE_WSMOTER_TABULAR:
        train_local = wsmoter_oversample_regression(
            df=train_base,
            feature_cols=WSMOTER_FEATURE_COLS,
            target_col=TARGET_COL,
            id_col="ID",
            weight_col="Gradient Weight",
            multiplier=hparams["wsmoter_multiplier"],
            k_neighbors=WSMOTER_K_NEIGHBORS,
            weight_power=hparams["wsmoter_weight_power"],
            random_state=SEED,
            id_mode=WSMOTER_ID_MODE
        )
    else:
        train_local = train_base.copy()
        train_local["is_synthetic"] = 0
        train_local["source_ID"] = train_local["ID"].astype(int)

    val_local = val_base.copy()
    val_local["is_synthetic"] = 0
    val_local["source_ID"] = val_local["ID"].astype(int)
    val_local = val_local.drop(columns=["Gradient Weight"], errors="ignore")

    train_original_local["is_synthetic"] = 0
    train_original_local["source_ID"] = train_original_local["ID"].astype(int)

    if include_test:
        test_local = test_base.copy()
        test_local["is_synthetic"] = 0
        test_local["source_ID"] = test_local["ID"].astype(int)
        test_local = test_local.drop(columns=["Gradient Weight"], errors="ignore")
    else:
        test_local = None

    dataframes_for_target = [train_local, train_original_local, val_local]
    if include_test:
        dataframes_for_target.append(test_local)

    for _df in dataframes_for_target:
        _df[TARGET_COL + "_raw"] = _df[TARGET_COL].copy()

    min_train_target = train_original_local[TARGET_COL].min()
    target_shift_local = 0.0

    if min_train_target <= -1:
        target_shift_local = float(abs(min_train_target) + 1.0)
        for _df in dataframes_for_target:
            _df[TARGET_COL] += target_shift_local

    for _df in dataframes_for_target:
        _df[TARGET_COL] = np.log1p(_df[TARGET_COL].astype(float))

    scaler_y_local = StandardScaler()
    train_local[[TARGET_COL]] = scaler_y_local.fit_transform(train_local[[TARGET_COL]])
    train_original_local[[TARGET_COL]] = scaler_y_local.transform(train_original_local[[TARGET_COL]])
    val_local[[TARGET_COL]] = scaler_y_local.transform(val_local[[TARGET_COL]])
    if include_test:
        test_local[[TARGET_COL]] = scaler_y_local.transform(test_local[[TARGET_COL]])

    target_scaler = scaler_y_local
    target_shift = target_shift_local

    if persist:
        joblib.dump(scaler_tab_local, OUT_DIR / "tabular_scaler_best_optuna.pkl")
        joblib.dump(scaler_y_local, OUT_DIR / "target_scaler_best_optuna.pkl")
        joblib.dump({"shift": target_shift_local}, OUT_DIR / "target_shift_best_optuna.json")
        train_local.to_csv(OUT_DIR / "train_wsmoter_best_optuna.csv", index=False)
        train_original_local.to_csv(OUT_DIR / "train_original_best_optuna.csv", index=False)
        val_local.to_csv(OUT_DIR / "val_best_optuna.csv", index=False)
        if include_test:
            test_local.to_csv(OUT_DIR / "test_best_optuna.csv", index=False)

    return train_local, train_original_local, val_local, test_local, scaler_y_local, target_shift_local


def build_loaders_for_hparams(train_local, train_original_local, val_local, test_local, hparams):
    train_augment_local = MultibandPairAugment(
        p_hflip=hparams["aug_p_hflip"],
        p_vflip=hparams["aug_p_vflip"],
        p_rot90=hparams["aug_p_rot90"],
        p_noise=hparams["aug_p_noise"],
        noise_sigma_image=hparams["aug_noise_sigma_image"],
        noise_sigma_thermal=hparams["aug_noise_sigma_thermal"],
        clip_value=AUG_CLIP_VALUE
    ) if hparams["use_image_augmentation"] else None

    train_dataset_local = MultimodalDataset(train_local, train=True, augment=train_augment_local)
    train_eval_dataset_local = MultimodalDataset(train_original_local, train=False, augment=None)
    val_dataset_local = MultimodalDataset(val_local, train=False, augment=None)

    batch_size = hparams["batch_size"]
    common = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

    loaders = {
        "train": DataLoader(
            train_dataset_local,
            batch_size=batch_size,
            shuffle=True,
            drop_last=True,
            **common
        ),
        "train_eval": DataLoader(
            train_eval_dataset_local,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            **common
        ),
        "val": DataLoader(
            val_dataset_local,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            **common
        ),
        "test": None
    }

    if test_local is not None:
        test_dataset_local = MultimodalDataset(test_local, train=False, augment=None)
        loaders["test"] = DataLoader(
            test_dataset_local,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            **common
        )

    return loaders


def build_model_from_hparams(hparams):
    return ConvNeXt9_Multimodal_BiAttn_Residual(
        n_tabular=len(tabular_cols),
        n_image_channels=N_IMAGE_CHANNELS,
        n_thermal_channels=N_THERMAL_DERIVED_CHANNELS,
        backbone_name=CONVNEXT_BACKBONE,
        pretrained=PRETRAINED_CONVNEXT,
        d_model=hparams["d_model"],
        n_heads=hparams["n_heads"],
        dropout=hparams["dropout"],
        token_pool_size=tuple(hparams["token_pool_size"]),
        img_token_layers=hparams["img_token_layers"],
        thermal_token_layers=hparams["thermal_token_layers"],
        img_dim_feedforward=hparams["img_dim_feedforward"],
        thermal_dim_feedforward=hparams["thermal_dim_feedforward"],
        tab_hidden=hparams["tab_hidden"],
        head_hidden_1=hparams["head_hidden_1"],
        head_hidden_2=hparams["head_hidden_2"],
        se_fusion_reduction=hparams["se_fusion_reduction"],
        spatial_attention_kernel=hparams["spatial_attention_kernel"]
    ).to(DEVICE)


def find_best_checkpoint(ckpt_dir):
    candidates = []
    for name in ["best_head.pth", "best_backbone.pth", "best_full.pth"]:
        path = Path(ckpt_dir) / name
        if path.exists():
            ck = torch.load(path, map_location="cpu", weights_only=False)
            candidates.append((float(ck.get("val_rmse_real", np.inf)), path, ck.get("stage", name)))

    if len(candidates) == 0:
        raise FileNotFoundError(f"No se encontró ningún checkpoint en {ckpt_dir}")

    candidates = sorted(candidates, key=lambda x: x[0])
    return candidates[0][1], candidates[0][0], candidates[0][2]


def fit_staged_model(model, loaders, criterion, hparams, ckpt_dir, trial=None):
    history_all = []
    step_offset = 0

    history_head = train_phase(
        model=model,
        train_loader=loaders["train"],
        train_eval_loader=loaders["train_eval"],
        val_loader=loaders["val"],
        criterion=criterion,
        epochs=hparams["epochs_head"],
        stage_name="head",
        ckpt_name="best_head.pth",
        hparams=hparams,
        ckpt_dir=ckpt_dir,
        trial=trial,
        step_offset=step_offset
    )
    history_all.extend(history_head)
    step_offset += hparams["epochs_head"]

    ck = torch.load(Path(ckpt_dir) / "best_head.pth", map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"])

    history_backbone = train_phase(
        model=model,
        train_loader=loaders["train"],
        train_eval_loader=loaders["train_eval"],
        val_loader=loaders["val"],
        criterion=criterion,
        epochs=hparams["epochs_backbone"],
        stage_name="backbone",
        ckpt_name="best_backbone.pth",
        hparams=hparams,
        ckpt_dir=ckpt_dir,
        trial=trial,
        step_offset=step_offset
    )
    history_all.extend(history_backbone)
    step_offset += hparams["epochs_backbone"]

    if hparams["epochs_full"] > 0:
        best_so_far_path, _, _ = find_best_checkpoint(ckpt_dir)
        ck = torch.load(best_so_far_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"])

        history_full = train_phase(
            model=model,
            train_loader=loaders["train"],
            train_eval_loader=loaders["train_eval"],
            val_loader=loaders["val"],
            criterion=criterion,
            epochs=hparams["epochs_full"],
            stage_name="full",
            ckpt_name="best_full.pth",
            hparams=hparams,
            ckpt_dir=ckpt_dir,
            trial=trial,
            step_offset=step_offset
        )
        history_all.extend(history_full)

    best_path, best_val_rmse, best_stage = find_best_checkpoint(ckpt_dir)
    pd.DataFrame(history_all).to_csv(Path(ckpt_dir) / "history.csv", index=False)

    return history_all, best_path, best_val_rmse, best_stage


def objective(trial):
    global target_scaler, target_shift

    hparams = sample_hparams(trial)
    trial.set_user_attr("resolved_hparams", _jsonable_hparams(hparams))

    trial_dir = OPTUNA_TMP_DIR / f"trial_{trial.number:03d}"
    trial_dir.mkdir(parents=True, exist_ok=True)

    model_trial = None
    loaders_trial = None

    try:
        seed_everything(SEED + trial.number)

        train_local, train_original_local, val_local, _, scaler_y_local, shift_local = prepare_dataframes_for_hparams(
            hparams,
            include_test=False,
            persist=False
        )

        loaders_trial = build_loaders_for_hparams(
            train_local=train_local,
            train_original_local=train_original_local,
            val_local=val_local,
            test_local=None,
            hparams=hparams
        )

        model_trial = build_model_from_hparams(hparams)
        criterion_trial = torch.nn.HuberLoss(delta=hparams["huber_delta"])

        _, best_path, best_val_rmse, best_stage = fit_staged_model(
            model=model_trial,
            loaders=loaders_trial,
            criterion=criterion_trial,
            hparams=hparams,
            ckpt_dir=trial_dir,
            trial=trial
        )

        trial.set_user_attr("best_checkpoint", str(best_path))
        trial.set_user_attr("best_stage", best_stage)
        trial.set_user_attr("target_shift", shift_local)

        return best_val_rmse

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            raise optuna.TrialPruned("CUDA out of memory")
        raise

    finally:
        del model_trial
        del loaders_trial
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

class KeepTopKCheckpoints:
    def __init__(self, top_k, top_k_dir, tmp_dir):
        self.top_k = int(top_k)
        self.top_k_dir = Path(top_k_dir)
        self.tmp_dir = Path(tmp_dir)
        self.top_k_dir.mkdir(parents=True, exist_ok=True)

    def _read_ckpt_metadata(self, ckpt_path):
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)

        stage = ck.get("stage", "unknown")
        epoch = ck.get("epoch", "unknown")
        rmse = ck.get("val_rmse_real", None)

        # Si tus épocas son 0-indexed y quieres nombres humanos 1-indexed,
        # cambia esto a: epoch = int(epoch) + 1
        if isinstance(epoch, int):
            epoch_str = f"{epoch:03d}"
        else:
            epoch_str = str(epoch)

        return stage, epoch_str, rmse

    def __call__(self, study, frozen_trial):
        trial_dir = self.tmp_dir / f"trial_{frozen_trial.number:03d}"

        try:
            complete_trials = [
                t for t in study.trials
                if t.state == optuna.trial.TrialState.COMPLETE
                and t.value is not None
            ]

            complete_trials = sorted(complete_trials, key=lambda t: float(t.value))
            top_trials = complete_trials[:self.top_k]
            top_trial_numbers = {t.number for t in top_trials}

            # Copiar el checkpoint del trial actual si quedó dentro del top K.
            if (
                frozen_trial.state == optuna.trial.TrialState.COMPLETE
                and frozen_trial.value is not None
                and frozen_trial.number in top_trial_numbers
            ):
                best_ckpt = frozen_trial.user_attrs.get("best_checkpoint")

                if best_ckpt is not None and Path(best_ckpt).exists():
                    best_ckpt = Path(best_ckpt)

                    stage, epoch_str, ckpt_rmse = self._read_ckpt_metadata(best_ckpt)

                    # Usa el RMSE del checkpoint si existe; si no, usa el valor del trial.
                    value = float(ckpt_rmse) if ckpt_rmse is not None else float(frozen_trial.value)

                    dst_name = (
                        f"trial_{frozen_trial.number:03d}"
                        f"_stage_{stage}"
                        f"_epoch_{epoch_str}"
                        f"_rmse_{value:.6f}.pth"
                    )

                    dst_path = self.top_k_dir / dst_name

                    # Eliminar versiones previas del mismo trial.
                    for old in self.top_k_dir.glob(f"trial_{frozen_trial.number:03d}_stage_*_epoch_*_rmse_*.pth"):
                        old.unlink(missing_ok=True)

                    shutil.copy2(best_ckpt, dst_path)

                    print(f"[Optuna] Checkpoint top-{self.top_k} guardado: {dst_path}")

            # Eliminar checkpoints persistentes de trials que ya salieron del top K.
            for ckpt_path in self.top_k_dir.glob("trial_*_stage_*_epoch_*_rmse_*.pth"):
                try:
                    trial_num = int(ckpt_path.name.split("_")[1])
                except Exception:
                    continue

                if trial_num not in top_trial_numbers:
                    ckpt_path.unlink(missing_ok=True)
                    print(f"[Optuna] Eliminado del top-{self.top_k}: {ckpt_path}")

            # Crear manifiesto actualizado.
            manifest = []

            for rank, t in enumerate(top_trials, start=1):
                matches = list(
                    self.top_k_dir.glob(
                        f"trial_{t.number:03d}_stage_*_epoch_*_rmse_*.pth"
                    )
                )

                checkpoint_path = str(matches[0]) if matches else None

                manifest.append({
                    "rank": rank,
                    "trial": int(t.number),
                    "value": float(t.value),
                    "checkpoint": checkpoint_path,
                    "best_stage": t.user_attrs.get("best_stage"),
                    "resolved_hparams": t.user_attrs.get("resolved_hparams"),
                })

            with open(self.top_k_dir / "top5_manifest.json", "w", encoding="utf-8") as fh:
                json.dump(manifest, fh, indent=2, ensure_ascii=False)

        finally:
            # Borra siempre la carpeta temporal del trial actual.
            if trial_dir.exists():
                shutil.rmtree(trial_dir, ignore_errors=True)

if USE_OPTUNA:
    sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)

    study = optuna.create_study(
        direction="minimize",
        study_name=OPTUNA_STUDY_NAME,
        sampler=sampler,
        pruner=pruner
    )

    study.optimize(
        objective,
        n_trials=OPTUNA_N_TRIALS,
        timeout=OPTUNA_TIMEOUT,
        gc_after_trial=True,
        show_progress_bar=True,
        callbacks=[
        KeepTopKCheckpoints(
            top_k=TOP_K,
            top_k_dir=TOP_K_CKPT_DIR,
            tmp_dir=OPTUNA_TMP_DIR
        )
        ]
    )

    print("\n===== MEJOR TRIAL OPTUNA =====")
    print("Trial:", study.best_trial.number)
    print("ValRMSE_real:", study.best_value)
    print("Mejor stage:", study.best_trial.user_attrs.get("best_stage"))
    print("Checkpoint:", study.best_trial.user_attrs.get("best_checkpoint"))
    print("Hiperparámetros resueltos:")
    print(json.dumps(study.best_trial.user_attrs["resolved_hparams"], indent=2, ensure_ascii=False))

    study.trials_dataframe().to_csv(OUT_DIR / "optuna_trials.csv", index=False)
    with open(OUT_DIR / "best_hparams_optuna.json", "w", encoding="utf-8") as fh:
        json.dump(study.best_trial.user_attrs["resolved_hparams"], fh, indent=2, ensure_ascii=False)
else:
    print("USE_OPTUNA=False. Se omite la búsqueda de hiperparámetros.")


In [ ]:
# Cell 10: evaluación final con el mejor modelo de Optuna
# Aquí sí se construye test_loader y se evalúa test una sola vez.

if not USE_OPTUNA:
    raise RuntimeError("Activa USE_OPTUNA=True para usar esta celda.")

best_hparams = dict(study.best_trial.user_attrs["resolved_hparams"])
best_hparams["token_pool_size"] = tuple(best_hparams["token_pool_size"])

def resolve_best_optuna_checkpoint(study):
    """
    Busca el checkpoint real del mejor trial.
    Primero intenta la ruta original.
    Si fue borrada de /kaggle/temp, busca en TOP_K_CKPT_DIR.
    """
    best_trial_num = study.best_trial.number

    # 1. Ruta original guardada por Optuna
    original_path = Path(study.best_trial.user_attrs["best_checkpoint"])
    if original_path.exists():
        return original_path

    print("Checkpoint temporal no encontrado:")
    print(original_path)
    print("Buscando checkpoint persistente en TOP_K_CKPT_DIR...")

    # 2. Buscar en el manifiesto top5
    manifest_path = TOP_K_CKPT_DIR / "top5_manifest.json"

    if manifest_path.exists():
        with open(manifest_path, "r", encoding="utf-8") as fh:
            manifest = json.load(fh)

        for row in manifest:
            if int(row["trial"]) == int(best_trial_num):
                ckpt = row.get("checkpoint", None)
                if ckpt is not None:
                    ckpt = Path(ckpt)
                    if ckpt.exists():
                        print("Checkpoint encontrado desde manifest:")
                        print(ckpt)
                        return ckpt

    # 3. Buscar directamente por nombre de trial
    matches = sorted(
        TOP_K_CKPT_DIR.glob(f"trial_{best_trial_num:03d}_stage_*_epoch_*_rmse_*.pth")
    )

    if len(matches) > 0:
        def get_rmse_from_name(path):
            m = re.search(r"_rmse_([0-9.]+)\.pth$", path.name)
            return float(m.group(1)) if m else float("inf")

        best_match = min(matches, key=get_rmse_from_name)
        print("Checkpoint encontrado por búsqueda directa:")
        print(best_match)
        return best_match

    # 4. Diagnóstico si no aparece
    available_ckpts = list(TOP_K_CKPT_DIR.glob("*.pth"))

    msg = (
        f"No se encontró checkpoint para el mejor trial {best_trial_num}.\n\n"
        f"Ruta original borrada:\n{original_path}\n\n"
        f"Carpeta persistente revisada:\n{TOP_K_CKPT_DIR}\n\n"
        f"Checkpoints disponibles:\n"
    )

    if len(available_ckpts) == 0:
        msg += "No hay checkpoints .pth en TOP_K_CKPT_DIR."
    else:
        msg += "\n".join(str(p) for p in available_ckpts)

    raise FileNotFoundError(msg)


best_ckpt_path = resolve_best_optuna_checkpoint(study)

# Opcional pero recomendado: copiar el mejor checkpoint a una ruta fija
persistent_best_ckpt = OUT_DIR / "best_checkpoint_optuna.pth"

if best_ckpt_path != persistent_best_ckpt:
    shutil.copy2(best_ckpt_path, persistent_best_ckpt)
    best_ckpt_path = persistent_best_ckpt

print("Checkpoint final que se usará:")
print(best_ckpt_path)

# Reconstruir exactamente los dataframes del mejor trial, ahora incluyendo test.
train_df, train_original_df, val_df, test_df, target_scaler, target_shift = prepare_dataframes_for_hparams(
    best_hparams,
    include_test=True,
    persist=True
)

loaders_best = build_loaders_for_hparams(
    train_local=train_df,
    train_original_local=train_original_df,
    val_local=val_df,
    test_local=test_df,
    hparams=best_hparams
)

train_loader = loaders_best["train"]
train_eval_loader = loaders_best["train_eval"]
val_loader = loaders_best["val"]
test_loader = loaders_best["test"]

model = build_model_from_hparams(best_hparams)
criterion = torch.nn.HuberLoss(delta=best_hparams["huber_delta"])

ck = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model"])
print("Checkpoint Optuna cargado:", best_ckpt_path)
print("Stage:", ck.get("stage", "unk"), "epoch:", ck.get("epoch", "unk"))
print("ValRMSE_real guardado:", ck.get("val_rmse_real", "unk"))

# TRAIN ORIGINAL, sin WSMOTER y sin augmentación
train_eval = evaluate_loader(model, train_eval_loader, criterion)
train_preds_real = train_eval["preds_real"]
train_targs_real = train_eval["targets_real"]
ids_train = train_eval["ids"]

train_rmse = train_eval["rmse_real"]
train_mae = train_eval["mae_real"]
train_r2 = train_eval["r2_real"]

# VALIDACIÓN
val_eval = evaluate_loader(model, val_loader, criterion)
val_preds_real = val_eval["preds_real"]
val_targs_real = val_eval["targets_real"]
ids_val = val_eval["ids"]

val_rmse = val_eval["rmse_real"]
val_mae = val_eval["mae_real"]
val_r2 = val_eval["r2_real"]

# TEST: se usa solo después de elegir los mejores hiperparámetros.
test_eval = evaluate_loader(model, test_loader, criterion)
test_preds_real = test_eval["preds_real"]
test_targs_real = test_eval["targets_real"]
ids_test = test_eval["ids"]

test_rmse = test_eval["rmse_real"]
test_mae = test_eval["mae_real"]
test_r2 = test_eval["r2_real"]

print("\n===== TRAIN ORIGINAL, sin sintéticos (unidades reales: °C/Km) =====")
print("RMSE: %.4f  MAE: %.4f  R2: %.4f" % (train_rmse, train_mae, train_r2))

print("\n===== VALIDATION (unidades reales: °C/Km) =====")
print("RMSE: %.4f  MAE: %.4f  R2: %.4f" % (val_rmse, val_mae, val_r2))

print("\n===== TEST FINAL OPTUNA (unidades reales: °C/Km) =====")
print("RMSE: %.4f  MAE: %.4f  R2: %.4f" % (test_rmse, test_mae, test_r2))

final_metrics = {
    "train_rmse": float(train_rmse),
    "train_mae": float(train_mae),
    "train_r2": float(train_r2),
    "val_rmse": float(val_rmse),
    "val_mae": float(val_mae),
    "val_r2": float(val_r2),
    "test_rmse": float(test_rmse),
    "test_mae": float(test_mae),
    "test_r2": float(test_r2),
    "best_checkpoint": str(best_ckpt_path),
    "best_hparams": _jsonable_hparams(best_hparams)
}

with open(OUT_DIR / "final_metrics_best_optuna.json", "w", encoding="utf-8") as fh:
    json.dump(final_metrics, fh, indent=2, ensure_ascii=False)


In [ ]:
# ============================================================
# Cell 10: evaluación final de los 5 mejores modelos de Optuna
# Evalúa TRAIN ORIGINAL, VALIDATION y TEST para cada modelo
# ============================================================

if not USE_OPTUNA:
    raise RuntimeError("Activa USE_OPTUNA=True para usar esta celda.")

import json
import re
import shutil
import gc
from pathlib import Path

import pandas as pd
import torch

try:
    from optuna.trial import TrialState
except Exception:
    TrialState = None


# ------------------------------------------------------------
# Utilidad para convertir hiperparámetros a formato JSON
# ------------------------------------------------------------
def jsonable_hparams_local(hparams):
    out = {}
    for k, v in hparams.items():
        if isinstance(v, tuple):
            out[k] = list(v)
        elif isinstance(v, (int, float, str, bool)) or v is None:
            out[k] = v
        else:
            try:
                out[k] = float(v)
            except Exception:
                out[k] = str(v)
    return out


# ------------------------------------------------------------
# Resolver checkpoint para cualquier trial, no solo el mejor
# ------------------------------------------------------------
def resolve_optuna_checkpoint_for_trial(study, trial):
    """
    Busca el checkpoint real de un trial de Optuna.
    Primero intenta la ruta original.
    Si fue borrada de /kaggle/temp, busca en TOP_K_CKPT_DIR.
    """

    trial_num = trial.number

    # 1. Ruta original guardada por Optuna
    original_ckpt = trial.user_attrs.get("best_checkpoint", None)

    if original_ckpt is not None:
        original_path = Path(original_ckpt)
        if original_path.exists():
            return original_path

        print(f"\nCheckpoint temporal no encontrado para trial {trial_num}:")
        print(original_path)
        print("Buscando checkpoint persistente en TOP_K_CKPT_DIR...")

    # 2. Buscar en el manifiesto top5
    manifest_path = TOP_K_CKPT_DIR / "top5_manifest.json"

    if manifest_path.exists():
        with open(manifest_path, "r", encoding="utf-8") as fh:
            manifest = json.load(fh)

        for row in manifest:
            if int(row["trial"]) == int(trial_num):
                ckpt = row.get("checkpoint", None)
                if ckpt is not None:
                    ckpt = Path(ckpt)
                    if ckpt.exists():
                        print(f"Checkpoint encontrado desde manifest para trial {trial_num}:")
                        print(ckpt)
                        return ckpt

    # 3. Buscar directamente por nombre de trial
    matches = sorted(
        TOP_K_CKPT_DIR.glob(f"trial_{trial_num:03d}_stage_*_epoch_*_rmse_*.pth")
    )

    if len(matches) > 0:

        def get_rmse_from_name(path):
            m = re.search(r"_rmse_([0-9.]+)\.pth$", path.name)
            return float(m.group(1)) if m else float("inf")

        best_match = min(matches, key=get_rmse_from_name)

        print(f"Checkpoint encontrado por búsqueda directa para trial {trial_num}:")
        print(best_match)

        return best_match

    # 4. Diagnóstico si no aparece
    available_ckpts = list(TOP_K_CKPT_DIR.glob("*.pth"))

    msg = (
        f"No se encontró checkpoint para el trial {trial_num}.\n\n"
        f"Carpeta persistente revisada:\n{TOP_K_CKPT_DIR}\n\n"
        f"Checkpoints disponibles:\n"
    )

    if len(available_ckpts) == 0:
        msg += "No hay checkpoints .pth en TOP_K_CKPT_DIR."
    else:
        msg += "\n".join(str(p) for p in available_ckpts)

    raise FileNotFoundError(msg)


# ------------------------------------------------------------
# Obtener los 5 mejores trials completos de Optuna
# ------------------------------------------------------------
if TrialState is not None:
    completed_trials = [
        t for t in study.trials
        if t.state == TrialState.COMPLETE and t.value is not None
    ]
else:
    completed_trials = [
        t for t in study.trials
        if t.value is not None
    ]

top5_trials = sorted(completed_trials, key=lambda t: t.value)[:5]

if len(top5_trials) == 0:
    raise RuntimeError("No hay trials completos con valor objetivo en el estudio.")

print(f"Número de trials completos encontrados: {len(completed_trials)}")
print(f"Número de modelos que se evaluarán: {len(top5_trials)}")


# ------------------------------------------------------------
# Evaluar TRAIN, VALIDATION y TEST para cada uno de los top 5
# ------------------------------------------------------------
all_metrics = []
all_predictions = []

for rank, trial in enumerate(top5_trials, start=1):

    print("\n" + "=" * 80)
    print(f"EVALUANDO MODELO TOP {rank}")
    print(f"Trial: {trial.number}")
    print(f"Valor objetivo Optuna: {trial.value}")
    print("=" * 80)

    # -----------------------------
    # Hiperparámetros del trial
    # -----------------------------
    hparams = dict(trial.user_attrs["resolved_hparams"])

    if "token_pool_size" in hparams:
        hparams["token_pool_size"] = tuple(hparams["token_pool_size"])

    # -----------------------------
    # Resolver checkpoint
    # -----------------------------
    ckpt_path = resolve_optuna_checkpoint_for_trial(study, trial)

    # Copia persistente opcional
    persistent_ckpt = OUT_DIR / f"top{rank}_trial_{trial.number:03d}_checkpoint_optuna.pth"

    if ckpt_path != persistent_ckpt:
        shutil.copy2(ckpt_path, persistent_ckpt)
        ckpt_path = persistent_ckpt

    print("Checkpoint que se usará:")
    print(ckpt_path)

    # -----------------------------
    # Reconstruir dataframes del trial
    # Incluye test solamente para evaluación final
    # -----------------------------
    train_df, train_original_df, val_df, test_df, target_scaler, target_shift = prepare_dataframes_for_hparams(
        hparams,
        include_test=True,
        persist=False
    )

    loaders = build_loaders_for_hparams(
        train_local=train_df,
        train_original_local=train_original_df,
        val_local=val_df,
        test_local=test_df,
        hparams=hparams
    )

    train_eval_loader = loaders["train_eval"]   # train original, sin sintéticos
    val_loader = loaders["val"]
    test_loader = loaders["test"]

    # -----------------------------
    # Construir y cargar modelo
    # -----------------------------
    model = build_model_from_hparams(hparams)
    model = model.to(DEVICE)

    criterion = torch.nn.HuberLoss(delta=hparams["huber_delta"])

    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"])
    model.eval()

    print("Checkpoint cargado correctamente")
    print("Stage:", ck.get("stage", "unk"), "epoch:", ck.get("epoch", "unk"))
    print("ValRMSE_real guardado:", ck.get("val_rmse_real", "unk"))

    # -----------------------------
    # TRAIN ORIGINAL
    # -----------------------------
    train_eval = evaluate_loader(model, train_eval_loader, criterion)

    train_rmse = train_eval["rmse_real"]
    train_mae = train_eval["mae_real"]
    train_r2 = train_eval["r2_real"]

    # -----------------------------
    # VALIDACIÓN
    # -----------------------------
    val_eval = evaluate_loader(model, val_loader, criterion)

    val_rmse = val_eval["rmse_real"]
    val_mae = val_eval["mae_real"]
    val_r2 = val_eval["r2_real"]

    # -----------------------------
    # TEST
    # -----------------------------
    test_eval = evaluate_loader(model, test_loader, criterion)

    test_rmse = test_eval["rmse_real"]
    test_mae = test_eval["mae_real"]
    test_r2 = test_eval["r2_real"]

    # -----------------------------
    # Mostrar resultados del modelo actual
    # -----------------------------
    print("\n===== TRAIN ORIGINAL, sin sintéticos, unidades reales: °C/Km =====")
    print("RMSE: %.4f  MAE: %.4f  R2: %.4f" % (train_rmse, train_mae, train_r2))

    print("\n===== VALIDATION, unidades reales: °C/Km =====")
    print("RMSE: %.4f  MAE: %.4f  R2: %.4f" % (val_rmse, val_mae, val_r2))

    print("\n===== TEST FINAL, unidades reales: °C/Km =====")
    print("RMSE: %.4f  MAE: %.4f  R2: %.4f" % (test_rmse, test_mae, test_r2))

    # -----------------------------
    # Guardar métricas
    # -----------------------------
    row_metrics = {
        "rank": int(rank),
        "trial": int(trial.number),
        "optuna_objective_value": float(trial.value),

        "train_rmse": float(train_rmse),
        "train_mae": float(train_mae),
        "train_r2": float(train_r2),

        "val_rmse": float(val_rmse),
        "val_mae": float(val_mae),
        "val_r2": float(val_r2),

        "test_rmse": float(test_rmse),
        "test_mae": float(test_mae),
        "test_r2": float(test_r2),

        "stage": ck.get("stage", "unk"),
        "epoch": ck.get("epoch", "unk"),
        "checkpoint": str(ckpt_path),

        "hparams": jsonable_hparams_local(hparams)
    }

    all_metrics.append(row_metrics)

    # -----------------------------
    # Guardar predicciones por split
    # -----------------------------
    split_outputs = {
        "train_original": train_eval,
        "validation": val_eval,
        "test": test_eval
    }

    for split_name, ev in split_outputs.items():
        ids = ev["ids"]
        preds = ev["preds_real"]
        targets = ev["targets_real"]

        for sample_id, y_true, y_pred in zip(ids, targets, preds):
            all_predictions.append({
                "rank": int(rank),
                "trial": int(trial.number),
                "split": split_name,
                "id": sample_id,
                "target_real": float(y_true),
                "prediction_real": float(y_pred),
                "error": float(y_pred - y_true),
                "abs_error": float(abs(y_pred - y_true))
            })

    # Limpieza de memoria
    del model, ck
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# Tabla resumen de métricas
# ------------------------------------------------------------
metrics_df = pd.DataFrame([
    {
        "rank": m["rank"],
        "trial": m["trial"],
        "optuna_objective_value": m["optuna_objective_value"],

        "train_rmse": m["train_rmse"],
        "train_mae": m["train_mae"],
        "train_r2": m["train_r2"],

        "val_rmse": m["val_rmse"],
        "val_mae": m["val_mae"],
        "val_r2": m["val_r2"],

        "test_rmse": m["test_rmse"],
        "test_mae": m["test_mae"],
        "test_r2": m["test_r2"],

        "stage": m["stage"],
        "epoch": m["epoch"],
        "checkpoint": m["checkpoint"]
    }
    for m in all_metrics
])

print("\n" + "=" * 80)
print("RESUMEN FINAL DE LOS 5 MEJORES MODELOS")
print("=" * 80)

display(metrics_df)


# ------------------------------------------------------------
# Guardar resultados
# ------------------------------------------------------------
metrics_csv_path = OUT_DIR / "top5_metrics_optuna_train_val_test.csv"
metrics_json_path = OUT_DIR / "top5_metrics_optuna_train_val_test.json"
preds_csv_path = OUT_DIR / "top5_predictions_optuna_train_val_test.csv"

metrics_df.to_csv(metrics_csv_path, index=False)

with open(metrics_json_path, "w", encoding="utf-8") as fh:
    json.dump(all_metrics, fh, indent=2, ensure_ascii=False)

preds_df = pd.DataFrame(all_predictions)
preds_df.to_csv(preds_csv_path, index=False)

print("\nArchivos guardados:")
print(metrics_csv_path)
print(metrics_json_path)
print(preds_csv_path)

In [ ]:
# Cell 13: Mapas espaciales para test

def get_colombia_base():
    url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    world = gpd.read_file(url)

    possible_cols = ['iso_a3', 'ISO_A3', 'ADM0_A3', 'SOV_A3']
    code_col = next(c for c in possible_cols if c in world.columns)

    colombia = world[world[code_col] == 'COL']

    minx, miny, maxx, maxy = colombia.total_bounds
    padx = (maxx - minx) * 0.05
    pady = (maxy - miny) * 0.05
    bbox = (minx - padx, maxx + padx, miny - pady, maxy + pady)
    return colombia, bbox

def base_map(ax, colombia, bbox):
    colombia.boundary.plot(ax=ax, linewidth=1.0, color='black')
    ax.set_xlim(bbox[0], bbox[1])
    ax.set_ylim(bbox[2], bbox[3])
    ax.set_aspect('auto')
    ax.grid(True, linestyle='--', linewidth=0.3)

def get_geo_cols(df):
    """
    Devuelve las columnas de coordenadas sin normalizar.
    Se usan para mapas, no para la entrada normalizada del modelo.
    """
    lon_col = GEO_LON_COL if GEO_LON_COL in df.columns else "Longitude"
    lat_col = GEO_LAT_COL if GEO_LAT_COL in df.columns else "Latitude"
    return lon_col, lat_col

def make_prediction_geodataframe(df_geo, df_pred):
    df_geo = df_geo.copy()
    df_geo["ID"] = df_geo["ID"].astype(int)

    df_pred = df_pred.copy()
    df_pred["ID"] = df_pred["ID"].astype(int)

    merged = df_geo.merge(df_pred, on="ID", how="inner")
    lon_col, lat_col = get_geo_cols(merged)

    merged = gpd.GeoDataFrame(
        merged,
        geometry=gpd.points_from_xy(merged[lon_col], merged[lat_col]),
        crs="EPSG:4326"
    ).dropna(subset=["geometry"])

    return merged

def plot_prediction_maps(gdf, prefix):
    actual_col = "actual_real"
    pred_col = "pred_real"

    gdf = gdf.copy()
    gdf["diff"] = gdf[actual_col] - gdf[pred_col]
    gdf["abs_diff"] = np.abs(gdf["diff"])

    vmin = min(gdf[actual_col].min(), gdf[pred_col].min())
    vmax = max(gdf[actual_col].max(), gdf[pred_col].max())
    vabs = np.nanmax(np.abs(gdf["diff"]))
    vabs_abs = gdf["abs_diff"].max()

    colombia, bbox = get_colombia_base()

    # Actual
    fig, ax = plt.subplots(figsize=(10, 8))
    base_map(ax, colombia, bbox)
    gdf.plot(ax=ax, column=actual_col, cmap='jet', markersize=60, legend=True, vmin=vmin, vmax=vmax)
    fig.savefig(f"map_actual_{prefix}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    # Predicho
    fig, ax = plt.subplots(figsize=(10, 8))
    base_map(ax, colombia, bbox)
    gdf.plot(ax=ax, column=pred_col, cmap='jet', markersize=60, legend=True, vmin=vmin, vmax=vmax)
    fig.savefig(f"map_predicted_{prefix}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    # Diferencia
    fig, ax = plt.subplots(figsize=(10, 8))
    base_map(ax, colombia, bbox)
    gdf.plot(ax=ax, column="diff", cmap='RdBu_r', markersize=60, legend=True, vmin=-vabs, vmax=vabs)
    fig.savefig(f"map_difference_{prefix}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

    # Diferencia absoluta
    fig, ax = plt.subplots(figsize=(10, 8))
    base_map(ax, colombia, bbox)
    gdf.plot(ax=ax, column="abs_diff", cmap='viridis', markersize=60, legend=True, vmin=0, vmax=vabs_abs)
    fig.savefig(f"map_abs_difference_{prefix}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

df_pred_test = pd.DataFrame({
    "ID": ids_test,
    "actual_real": test_targs_real,
    "pred_real": test_preds_real
})

merged_test = make_prediction_geodataframe(test_df, df_pred_test)
plot_prediction_maps(merged_test, "TEST_REAL")


In [ ]:
# Cell 14: Mapas espaciales en entrenamiento

df_pred_train = pd.DataFrame({
    "ID": ids_train,
    "actual_real": train_targs_real,
    "pred_real": train_preds_real
})

# Para mapas de entrenamiento se usa train_original_df,
# no train_df con sintéticos WSMOTER.
merged_train = make_prediction_geodataframe(train_original_df, df_pred_train)
plot_prediction_maps(merged_train, "TRAIN_REAL")


In [ ]:
# Cell 15: Mapas espaciales todo el conjunto de datos (train + val + test)

df_pred_val = pd.DataFrame({
    "ID": ids_val,
    "actual_real": val_targs_real,
    "pred_real": val_preds_real
})

# El mapa completo usa solo muestras reales.
# No se incluyen las filas sintéticas de train_df.
full_geo_df = pd.concat(
    [train_original_df, val_df, test_df],
    ignore_index=True
).copy()

df_pred_all = pd.concat(
    [
        df_pred_train[["ID", "actual_real", "pred_real"]],
        df_pred_val[["ID", "actual_real", "pred_real"]],
        df_pred_test[["ID", "actual_real", "pred_real"]],
    ],
    ignore_index=True
)

merged_full = make_prediction_geodataframe(full_geo_df, df_pred_all)
plot_prediction_maps(merged_full, "FULL_REAL")


In [ ]:

# Cell 17: relación real vs predicho

tp = np.ravel(train_preds_real)
tt = np.ravel(train_targs_real)

vp = np.ravel(val_preds_real)
vt = np.ravel(val_targs_real)

ep = np.ravel(test_preds_real)
et = np.ravel(test_targs_real)

full_preds = np.concatenate([tp, vp, ep])
full_targs = np.concatenate([tt, vt, et])

datasets = {
    "TRAIN": (tp, tt),
    "VALIDATION": (vp, vt),
    "TEST": (ep, et),
    "FULL": (full_preds, full_targs)
}

all_vals = np.concatenate([tp, tt, vp, vt, ep, et])
xmin = np.nanmin(all_vals)
xmax = np.nanmax(all_vals)

pad = (xmax - xmin) * 0.06 if (xmax - xmin) > 0 else 0.5
xlim = (xmin - pad, xmax + pad)
ylim = (xmin - pad, xmax + pad)

point_color = "#1f4e5f"
line_color = "#F54927"
grid_color = "#d9d9d9"

for name, (preds, targs) in datasets.items():

    preds = np.ravel(preds)
    targs = np.ravel(targs)

    rmse = np.sqrt(mean_squared_error(targs, preds))
    mae = mean_absolute_error(targs, preds)
    r2 = r2_score(targs, preds) if len(targs) > 1 else np.nan

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.set_facecolor("#fafafa")

    ax.scatter(
        targs, preds,
        s=40,
        alpha=0.75,
        color=point_color,
        edgecolor="white",
        linewidth=0.4
    )

    ax.plot(
        [xlim[0], xlim[1]],
        [xlim[0], xlim[1]],
        linestyle="--",
        color=line_color,
        linewidth=1.6
    )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.set_xlabel("Gradiente real (°C/Km)", fontsize=12)
    ax.set_ylabel("Gradiente estimado (°C/Km)", fontsize=12)
    ax.set_title(f"{name}: real vs estimado", fontsize=13)

    ax.grid(True, linestyle="--", linewidth=0.6, color=grid_color)

    for spine in ax.spines.values():
        spine.set_color("#bbbbbb")
        spine.set_linewidth(0.8)

    fname = f"real_vs_estimado_{name.lower()}_REAL.pdf"
    fig.savefig(fname, dpi=300, bbox_inches='tight')

    print(
        f"{name} REAL -> RMSE={rmse:.3f}, "
        f"MAE={mae:.3f}, R2={r2:.3f}"
    )

    plt.show()


In [ ]:
# Cell 18: descargar mejor checkpoint y resultados de Optuna

import zipfile
from google.colab import files

zip_path = OUT_DIR / "best_optuna_model_and_results.zip"
files_to_zip = [
    best_ckpt_path,
    OUT_DIR / "best_hparams_optuna.json",
    OUT_DIR / "final_metrics_best_optuna.json",
    OUT_DIR / "optuna_trials.csv",
    OUT_DIR / "train_wsmoter_best_optuna.csv",
    OUT_DIR / "train_original_best_optuna.csv",
    OUT_DIR / "val_best_optuna.csv",
    OUT_DIR / "test_best_optuna.csv",
    OUT_DIR / "target_scaler_best_optuna.pkl",
    OUT_DIR / "target_shift_best_optuna.json",
    OUT_DIR / "tabular_scaler_best_optuna.pkl",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_zip:
        f = Path(f)
        if f.exists():
            zf.write(f, arcname=f.name)
        else:
            print("No encontrado, no se agrega al zip:", f)

print("Archivo creado:", zip_path)
files.download(str(zip_path))


# Cell 19
# best_head.zip fue reemplazado por best_optuna_model_and_results.zip.


In [ ]:
# Cell 20
# best_backbone.zip fue reemplazado por best_optuna_model_and_results.zip.
